## 📑 Core Assignment Data Mapping

#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/spaces/DBTC/pages/263225346/BRD+-+Payments+Transactional+Report)**  
  Comprehensive documentation outlining the business requirements for this report.

#####Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, regexp_replace, array, col, explode, lit, when, split

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import boto3
import os

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

som_date = spark.sql(
f'''
SELECT 
     MAX(cal.CalendarDate) CalendarDate
FROM `{env_prefix}-cdh`.reference.ref_calendar AS cal
WHERE cal.IsBusinessDay = 1
AND cal.CalendarDate <= LAST_DAY(DATE_ADD(MONTH, -1, DATE {run_date!r}))
''').collect()[0][0]

som_p_date = spark.sql(
f'''
SELECT 
     MAX(cal.CalendarDate) CalendarDate
FROM `{env_prefix}-cdh`.reference.ref_calendar AS cal
WHERE cal.IsBusinessDay = 1
AND cal.CalendarDate <= LAST_DAY(DATE_ADD(MONTH, -2, DATE {run_date!r}))
''').collect()[0][0]

referenceDateForHistorical = spark.sql(
f''' 
SELECT MAX(CASE WHEN (SELECT DayName
FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  MAKE_DATE(year({run_date!r}),month({run_date!r}),'14') ) = 'Saturday' 
																 THEN (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  MAKE_DATE(year({run_date!r}),month({run_date!r}),'16'))
																 ELSE (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  MAKE_DATE(year({run_date!r}),month({run_date!r}),'15')) 
													END	) CalendarDate
''')
referenceDateForHistorical.createOrReplaceTempView("referenceDateForHistorical")       

UniqueReferenceDate = spark.sql(
f''' 	   
SELECT CalendarDate ReferenceDate
FROM `{env_prefix}-cdh`.reference.ref_calendar
WHERE CalendarDate <= {run_date!r} AND CalendarDate >= {som_date!r}
and IsBusinessDay = 1
''')
UniqueReferenceDate.createOrReplaceTempView("UniqueReferenceDate")   

# Comment

##### Schema of the table

In [0]:
table_name = 'Core_AssignmentDataMapping'

schema = (
"BusinessDate date,"
"PortfolioName string,"
"ExposureKey string,"
"AgreementKey string,"
"CDI string,"
"PartyTaxRegistrationNumber string,"
"Entity string,"
"AssetSegment string,"
"ProductKey string,"
"SoMBalanceAmountEUR decimal(19,2),"
"SoMSourceBucket int,"
"SoMObjectiveDate date,"
"SoMObjectiveStatus string,"
"SoMDenouncedStatus string,"
"MtDQueues string,"
"MtDAgency string,"
"RiskBucket int,"
"EBAStatus string,"
"PENPEDebtor string,"
"AssignmentChannel string,"
"AssignmentTeam string,"
"AssignmentAgent string,"
"AssignmentAgentAM string,"
"Queues string,"
"CourseOfAction string,"
"AssignmentSegmentDetailed string,"
"Anchor string,"
"SavedFlag string,"
"SavedReason string,"
"SavedDate date,"
"SavedQueues string,"
"SavedAgency string,"
"Treatmentbranch string,"
"SubProduct string,"
"AttributionCOA string,"
"AttributionChannel string,"
"AttributionTeam string,"
"AttributionAgent string,"
"AttributionAgentAM string,"
"AttributionAgent_2ndLevel string,"
"AttributionAgentAM_2ndLevel string,"
"AttributionSegmentDetailed string,"
"Event string,"
"PaymentAmount decimal(19,2),"
"AttributionChannelOriginal string,"
"AttributionTeamOriginal string,"
"AttributionAgentOriginal string,"
"AttributionAgentAMOriginal string,"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Optimizer function

In [0]:
%python
# Helper function to create or replace table
def create_or_replace_table(table_name, df, partition_col):
    df.write.format("delta").mode("overwrite").partitionBy(partition_col).saveAsTable(table_name)
def create_or_replace_table_no_partition(table_name, df):
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
def repair_and_analyze_table(table_name):
    spark.sql(f"REPAIR TABLE {table_name} SYNC METADATA")
    spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS")    

##### SavedFile

In [0]:
SavedInfo = spark.sql(
f'''
SELECT DISTINCT 
       Account 
     , SavedReason
     , SavedFlag 
     , SavedDate 
from `{env_prefix}-cdh`.derivative.dportfoliotoolcore
--from `{env_prefix}-cdh-delivery`.repoint.dportfoliotoolcore_uat_27
WHERE businessdate={run_date!r}
AND RelationType = 'Primary Owner' 
''')
SavedInfo.createOrReplaceTempView("SavedInfo")

In [0]:
SavedFile = spark.sql(
f'''		
SELECT	s.*
	,	c.PreviousBusinessDay AS SavedDatePreviousBusinessDay
FROM SavedInfo s  
INNER JOIN `{env_prefix}-cdh`.reference.ref_calendar  c	
ON c.CalendarDate = s.SavedDate 
WHERE s.SavedFlag= 'Saved' 
''')
SavedFile.createOrReplaceTempView("SavedFile")  

##### MtDAccount & SoMAccount

In [0]:
CaseCode_Agency = spark.sql(
f'''
        SELECT  CaseCodeKey, Agency, nc.ExposureKey, c.BusinessDate, nc.CDI, Queue
        FROM `{env_prefix}-cdh`.primary.casecode c
        inner join `{env_prefix}-cdh`.primary.exposure nc on c.casecodekey = nc.casecode and nc.portfolioname = c.PortfolioName AND nc.BusinessDate=c.BusinessDate
        WHERE 1=1
        AND c.BusinessDate IN ({run_date!r},{som_date!r})
        and c.PortfolioName = 'Core Retail'
''')
CaseCode_Agency.createOrReplaceTempView("CaseCode_Agency")

In [0]:
product = spark.sql(
f'''
SELECT 
    ProductDescription,
    ProductKey,
    SourceSystem,
    BusinessDate
FROM `{env_prefix}-cdh`.primary.product
WHERE BusinessDate = {run_date!r}
''')

In [0]:
MtDAccount = spark.sql(
f'''
SELECT
	    de.ExposureKey
    , de.BusinessDate
    , de.PortfolioName
    , de.ExposureStatus
    , de.AgreementKey
    , de.CDI
    , de.PartyTaxRegistrationNumber 
    , IFNULL(de.PartyTaxRegistrationNumber,de.CDI) AS Entity
    , de.AssetSegment
    , IFNULL(e.Bucket, de.Bucket) AS Bucket
    , de.OutstandingTotalBalance
    , de.GBV
    , de.OffBalance
    , de.OnBalance
    , de.DueAmount
    , de.ObjectiveStatusCode   AS ObjectiveStatus
    , de.ObjectiveStatusdate
    , de.IsActiveFlag
    , de.SubProductDescription
    , cast(de.ProductCode AS int) AS SubProductCode
    , de.HasActiveSettlementFlag
    , de.SettlementBucket
    , de.ClosedDate
    , de.ExposureKeyQCAlpha
    , de.CaseCode
    , de.Currency
    , e.ProductKey
    , c.Queue
    , c.Agency 
    , CASE WHEN de.ExposureStatusCode = '074-DENOUNCED' THEN 'DENOUNCED'  
           WHEN de.ExposureStatusCode = '074-NON DELINQUENT' THEN 'NON DELINQUENT' 
           WHEN de.ExposureStatusCode = '074-DELINQUENT' THEN 'DELINQUENT' 
           WHEN de.ExposureStatusCode = '074-WRITE OFF' THEN 'WRITE OFF' 
           ELSE de.ExposureStatusCode END ExposureStatusCode
FROM `{env_prefix}-cdh`.derivative.dexposure de
INNER JOIN `{env_prefix}-cdh`.primary.exposure e ON e.ExposureKey = de.ExposureKey AND e.BusinessDate = de.BusinessDate
LEFT JOIN `{env_prefix}-cdh`.primary.casecode c on c.casecodekey = e.casecode and e.portfolioname = c.PortfolioName AND e.BusinessDate=c.BusinessDate
WHERE de.BusinessDate = {run_date!r}
  AND de.PortfolioName = 'Core Retail'
  AND e.PartyRelationshipType = 'Primary Owner' 
  AND de.IsActiveFlag = 'true'
'''
)
MtDAccount.createOrReplaceTempView("MtDAccount")

In [0]:
SoMAccount = spark.sql(
f'''
SELECT * 
FROM (
SELECT
      de.ExposureKey
    , de.BusinessDate
    , de.PortfolioName
    , de.ExposureStatus
    , de.AgreementKey
    , de.CDI
    , CASE WHEN de.PartyTaxRegistrationNumber='' THEN NULL ELSE de.PartyTaxRegistrationNumber END AS PartyTaxRegistrationNumber 
    , CASE WHEN de.PartyTaxRegistrationNumber='' or de.PartyTaxRegistrationNumber IS NULL THEN de.CDI ELSE de.PartyTaxRegistrationNumber END AS Entity
    , de.AssetSegment
    , IFNULL(e.Bucket, de.Bucket) AS Bucket
    , de.OutstandingTotalBalance
    , de.GBV
    , de.OffBalance
    , de.OnBalance
    , de.ClosedDate
    , de.DueAmount
    , de.ObjectiveStatusCode   AS ObjectiveStatus
    , de.IsActiveFlag
    , de.ObjectiveStatusdate
    , de.SubProductDescription
    , cast(de.ProductCode AS int) AS SubProductCode
    , de.HasActiveSettlementFlag
    , de.SettlementBucket
    , de.ExposureKeyQCAlpha
    , de.CaseCode
    , de.Currency
    , e.ProductKey
    , CASE WHEN de.ExposureStatusCode = '074-DENOUNCED' THEN 'DENOUNCED'  
           WHEN de.ExposureStatusCode = '074-NON DELINQUENT' THEN 'NON DELINQUENT' 
           WHEN de.ExposureStatusCode = '074-DELINQUENT' THEN 'DELINQUENT' 
           WHEN de.ExposureStatusCode = '074-WRITE OFF' THEN 'WRITE OFF' 
           ELSE de.ExposureStatusCode END ExposureStatusCode
    , CASE WHEN de.ObjectiveStatusCode rlike '[A-D][0-9]' THEN 'Anchor'
		   WHEN de.ObjectiveStatusCode ILIKE '%Overdraft%'  THEN 'Overdraft'
		   END  AS Anchor
    , ROW_NUMBER() OVER (PARTITION BY de.ExposureKey ORDER BY de.BusinessDate ASC) AS ranking
FROM `{env_prefix}-cdh`.derivative.dexposure de
INNER JOIN `{env_prefix}-cdh`.primary.exposure e ON e.ExposureKey = de.ExposureKey AND e.BusinessDate = de.BusinessDate
WHERE de.BusinessDate <= {run_date!r}
AND de.BusinessDate >= {som_date!r}
AND de.PortfolioName = 'Core Retail'
AND e.PartyRelationshipType = 'Primary Owner' 
)
WHERE ranking=1
''')
SoMAccount.createOrReplaceTempView("SoMAccount")

##### AssignedPerimeter

In [0]:
AssignedPerimeter = spark.sql(
f'''
		SELECT DISTINCT					
					IFNULL(MtD.PortfolioName, SoM.PortfolioName) AS PortfolioName
				,	IFNULL(MtD.ExposureKey, SoM.ExposureKey) AS ExposureKey
				,	IFNULL(MtD.AgreementKey, SoM.AgreementKey) AS AgreementKey
				,	IFNULL(MtD.CDI, SoM.CDI) AS CDI
				,	IFNULL(MtD.PartyTaxRegistrationNumber, SoM.PartyTaxRegistrationNumber) AS PartyTaxRegistrationNumber
				,	IFNULL(MtD.Entity, SoM.Entity) AS Entity
				,	IFNULL(MtD.AssetSegment, SoM.AssetSegment) AS AssetSegment
				,	IFNULL(MtD.ProductKey, SoM.ProductKey) AS ProductKey
				,	CAST(`{env_prefix}-cdh`.derivative.converttoeur(SoM.Currency, SoM.OutstandingTotalBalance, SoM.BusinessDate) AS DECIMAL(19,2)) SoMBalanceAmountEUR
				,	SoM.Bucket																AS SoMSourceBucket
				,	SoM.ExposureStatus												    AS SoMArrearsStatusENG
				,	SoM.ObjectiveStatusdate																AS SoMObjectiveDate
				,	SoM.ObjectiveStatus														  AS SoMObjectiveStatus
				,   SoM.ExposureStatusCode                                                   AS SoMDenouncedStatus 
                ,	MtD.IsActiveFlag																AS MtDIsActiveFlag
				,	MtD.ClosedDate																	AS MtDClosingDate
				,	MtD.Bucket																AS MtDSourceBucket
				,	MtD.Queue																		  AS MtDQueues
				,	MtD.Agency																	AS MtDAgency
				, SoM.Anchor 		
		FROM MtDAccount			  MtD			
		FULL JOIN SoMAccount SoM	ON SoM.ExposureKey = MtD.ExposureKey
''')
AssignedPerimeter.createOrReplaceTempView("AssignedPerimeter") 

create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_AssignedPerimeter", AssignedPerimeter)
repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_AssignedPerimeter")

##### DailyPortfolio

In [0]:
SubProduct = spark.sql(
f'''
SELECT DISTINCT ACCOUNT_NUMBER, SUBPRODUCT SubProduct
FROM(
SELECT DISTINCT A.ACCOUNT_NUMBER,A.SUBPRODUCT, rank()over(partition by ACCOUNT_NUMBER order by BusinessDate desc) ranking
FROM `{env_prefix}-cdh`.landed.repointqcab_daily_portfolio A
-- INNER JOIN AssignedPerimeter B ON A.ACCOUNT_NUMBER = B.ExposureKey
WHERE A.BusinessDate > {som_date!r} AND A.BusinessDate <= {run_date!r} and SUBPRODUCT is not null
)
WHERE ranking = 1
''')
SubProduct.createOrReplaceTempView("SubProduct") 

In [0]:
Treatmentbranch = spark.sql(
f'''
SELECT DISTINCT ACCOUNT_NUMBER, TREATMENT_BRANCH Treatmentbranch
FROM(
SELECT A.ACCOUNT_NUMBER,A.TREATMENT_BRANCH, rank()over(partition by ACCOUNT_NUMBER order by BusinessDate desc) ranking
FROM `{env_prefix}-cdh`.landed.repointqcab_daily_portfolio A
WHERE A.BusinessDate > {som_date!r} AND A.BusinessDate <= {run_date!r} AND TREATMENT_BRANCH IS NOT NULL
)
WHERE ranking = 1
''')
Treatmentbranch.createOrReplaceTempView("Treatmentbranch") 

##### UnionFile

In [0]:
last_day_som_date = spark.sql(f'''SELECT cast(last_day({som_date!r}) as string)''').collect()[0][0]
last_day_som_p_date = spark.sql(f'''SELECT cast(last_day({som_p_date!r}) as string)''').collect()[0][0]

UnionFile = spark.sql(
f'''			
SELECT 
			    U.ExposureNumber
			  ,	U.BucketCategoryWithMateriality AS RiskBucket
			  ,	CASE WHEN SnapshotDate = {last_day_som_date!r} THEN
                         CASE WHEN U.FPLUnderProbationFlag = 1 AND U.EBAStatus = 'FPL' THEN 'CFPL' 
                         ELSE U.EBAStatus END
               WHEN SnapshotDate = {last_day_som_p_date!r} THEN 
                         CASE WHEN U.FPLUnderProbationFlag = 0 AND U.EBAStatus = 'PL'   AND BucketCategoryWithMateriality IN (0,1,2,3) THEN 'PL' 
					                    WHEN U.FPLUnderProbationFlag = 0 AND U.EBAStatus = 'PL'   AND BucketCategoryWithMateriality > 4		    THEN 'NPL' 
					                    WHEN U.FPLUnderProbationFlag = 0 AND U.EBAStatus = 'FPL'  AND BucketCategoryWithMateriality IN (0,1,2,3) THEN 'FPL' 
					                    WHEN U.FPLUnderProbationFlag = 0 AND U.EBAStatus = 'FPL'  AND BucketCategoryWithMateriality > 4		    THEN 'FNPL' 
					                    WHEN U.FPLUnderProbationFlag = 1 AND U.EBAStatus = 'FPL'  AND BucketCategoryWithMateriality IN (0,1)	THEN 'CFPL' 
					                    WHEN U.FPLUnderProbationFlag = 1 AND U.EBAStatus = 'FPL'  AND BucketCategoryWithMateriality > 2		    THEN 'FNPL' 
					                    WHEN U.FPLUnderProbationFlag = 0 AND U.EBAStatus = 'FNPL' AND BucketCategoryWithMateriality > 0		    THEN 'FNPL' 
					                    WHEN U.FPLUnderProbationFlag = 0 AND U.EBAStatus = 'NPL'  AND BucketCategoryWithMateriality > 0		    THEN 'NPL' 					 
					                    ELSE U.EBAStatus END
           END EBAStatus
			  ,	U.FPLUnderProbationFlag
			  ,	U.SnapshotDate
FROM `{env_prefix}-cdh`.externalfiles.extunionfile       U
-- INNER JOIN (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extunionfile WHERE SnapshotDate IN(last_day({last_day_som_date!r}),last_day({last_day_som_p_date!r})) GROUP BY SnapshotDate) IT ON IT.IngestionTimeStamp = U.IngestionTimeStamp
-- WHERE SnapshotDate IN (last_day({last_day_som_date!r}),last_day({last_day_som_p_date!r}))
WHERE SnapshotDate = ifnull({last_day_som_date!r},{last_day_som_p_date!r})
''')
UnionFile.createOrReplaceTempView("UnionFile")

# Check if the dataset is empty and print a message accordingly
if UnionFile.count() == 0:
    print(f"The required external file extunionfile dataset for dates {last_day_som_date!r},{last_day_som_p_date!r} is empty")

##### AFMTotalEBA

In [0]:
AFMTotalEBA = spark.sql(
f'''	
SELECT Entity
	 , CASE WHEN PE_count > 0 AND NPE_Count >= 0 THEN 'PE'
			WHEN PE_count = 0 AND NPE_Count > 0  THEN 'NPE'
			END AS `PE/NPEDebtor`
FROM (
		        SELECT Entity
		        	 , SUM(CASE WHEN EBAStatus IN ('PL','CFPL','FLPL')  THEN 1 ELSE 0 END) AS PE_count
		        	 , SUM(CASE WHEN EBAStatus IN ('NPL','FNPL')		THEN 1 ELSE 0 END) AS NPE_count
		        FROM UnionFile u
				INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_AssignedPerimeter A ON u.ExposureNumber = A.ExposureKey
		        GROUP BY Entity 
			 ) a
''')
AFMTotalEBA.createOrReplaceTempView("AFMTotalEBA")           

##### Modifications

In [0]:
Modifications = spark.sql(
f'''	
SELECT * 
FROM (
		SELECT DISTINCT
		      		  OldExposureKey	    
		      	  ,	ModificationDate	
		      	  ,	Categoryfinal		
		      	  ,	RANK() OVER(PARTITION BY OldExposureKey ORDER BY ModificationDate ASC) AS Rank
		 FROM `{env_prefix}-cdh-delivery`.repoint.core_modifications 
		WHERE BusinessDate = {run_date!r}
	  ) a
WHERE a.Rank = 1
''')
Modifications.createOrReplaceTempView("Modifications")   

##### Queues

In [0]:
QueuesHistorical_Core = spark.sql(
f'''
SELECT DISTINCT     r.ReferenceDate 
				,	nc.ExposureKey 
				,	nc.CDI
		        ,   Queue Queues
				,	Agency					 
FROM `{env_prefix}-cdh`.primary.casecode c
inner join `{env_prefix}-cdh`.primary.exposure nc on c.casecodekey = nc.casecode and nc.portfolioname =  c.PortfolioName and nc.BusinessDate = c.BusinessDate
inner join UniqueReferenceDate r on r.ReferenceDate = c.BusinessDate
WHERE 1=1
AND c.BusinessDate <= {run_date!r} and c.BusinessDate >= {som_date!r}
and c.PortfolioName = 'Core Retail' 
AND PartyRelationshipType = 'Primary Owner'
''')
QueuesHistorical_Core.createOrReplaceTempView("QueuesHistorical_Core")  

In [0]:
# Queues = spark.sql(
# f'''	
# SELECT     allQ.*
# FROM QueuesHistorical_Core		allQ	
# INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_AssignedPerimeter acc ON ACC.ExposureKey = allQ.ExposureKey AND allQ.ReferenceDate < IFNULL(ACC.MtDClosingDate, '9999-01-01')
# ''')
# Queues.createOrReplaceTempView("Queues")  

create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_Queues", QueuesHistorical_Core)
repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_Queues")    

In [0]:
Queues_MaxDate = spark.sql(
f'''			
SELECT MAX(allQ.ReferenceDate) ReferenceDate
	,  allQ.ExposureKey
FROM `{env_prefix}-cdh-delivery`.repoint.int_Queues	allQ 
LEFT JOIN SavedFile	svf	ON svf.Account = allQ.ExposureKey AND svf.SavedDatePreviousBusinessDay <= (SELECT Distinct CalendarDate from referenceDateForHistorical)
WHERE 1=1 
	AND allQ.ReferenceDate <= (SELECT Distinct CalendarDate from referenceDateForHistorical)
	AND (svf.Account IS NULL OR svf.SavedDatePreviousBusinessDay > (SELECT Distinct CalendarDate from referenceDateForHistorical))
GROUP BY allQ.ExposureKey
''')
Queues_MaxDate.createOrReplaceTempView("Queues_MaxDate")    

##### Channels

In [0]:
AssignmentHistorical_Core = spark.sql(
f'''		
SELECT DISTINCT
  r.ReferenceDate  
, nc.ExposureKey   
, nc.CDI
, nc.CaseCode
, a.AssignmentType
, a.Channel AssignmentChannel	
, IFNULL(a.Team,'Unassigned') AssignmentTeam
,	a.AgentFullName AssignmentAgent 
 FROM `{env_prefix}-cdh`.primary.assignment A
 inner join `{env_prefix}-cdh`.primary.exposure nc on A.CaseCode = nc.casecode and nc.portfolioname = 'Core Retail' and nc.BusinessDate = A.BusinessDate
 inner join UniqueReferenceDate r on r.ReferenceDate = A.BusinessDate
 WHERE a.BusinessDate<={run_date!r} AND a.BusinessDate>={som_date!r}
   AND nc.PartyRelationshipType = 'Primary Owner'
   AND a.AssignmentType='Channel Assignment'
 ''')
AssignmentHistorical_Core.createOrReplaceTempView("AssignmentHistorical_Core")   

In [0]:
Channels = spark.sql(
f'''		
SELECT * 
FROM (
		      SELECT ch.* 
				   , RANK() OVER(PARTITION BY ch.ExposureKey, ch.ReferenceDate ORDER BY ch.ReferenceDate DESC, ch.AssignmentType ASC) AS Rank
		      FROM AssignmentHistorical_Core ch

	  		--   INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_AssignedPerimeter					acc	
			--   ON ACC.ExposureKey = ch.ExposureKey AND ch.ReferenceDate < IFNULL(ACC.MtDClosingDate, '9999-01-01')
	) a
WHERE a.Rank = 1
''')
Channels.createOrReplaceTempView("Channels")     

create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_Channels", Channels)
repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_Channels")

##### queuesmapping

In [0]:
queuesmapping = spark.sql(
f'''				
select * 
from `{env_prefix}-reportinputfiles`.repoint.core_queuesmapping
where BusinessDate = (SELECT MAX(BusinessDate) BusinessDate FROM `{env_prefix}-reportinputfiles`.repoint.core_queuesmapping WHERE BusinessDate <= {run_date!r})
''')
queuesmapping.createOrReplaceTempView("queuesmapping")  

##### assignmentlevelsmapping

In [0]:
assignmentlevelsmapping = spark.sql(
f'''				
select ReferenceDate
,1stlevelAssignment as `1st level Assignment`
,2ndlevelAssignment as `2nd level Assignment`
,3rdlevelAssignment as `3rd level Assignment`
,Lastname as `Last name1`
,Firstname as `First name`
,PENPERM as `PE/NPE RM`
,Fullname as `Full name`
,Fullname1 as `Full name 1`
,Fullname2 as `Full name 2`
,BusinessDate
,IngestionTimeStamp 
from `{env_prefix}-reportinputfiles`.repoint.core_assignmentlevelsmapping
where BusinessDate = (SELECT MAX(BusinessDate) BusinessDate FROM `{env_prefix}-reportinputfiles`.repoint.core_assignmentlevelsmapping WHERE BusinessDate <= {run_date!r})
''')
assignmentlevelsmapping.createOrReplaceTempView("assignmentlevelsmapping")  

##### QueuesChannels

In [0]:
QueuesChannels = spark.sql(
f'''				
  SELECT DISTINCT   Q.ExposureKey
				,	Q.ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	A.AssignmentChannel	
				,	A.AssignmentTeam
				,	A.AssignmentAgent
				,	lv.`3rd level Assignment` AS AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	svf.SavedFlag
				,	svf.SavedReason 
				,	svf.SavedDate
FROM `{env_prefix}-cdh-delivery`.repoint.int_Queues  Q
INNER JOIN Queues_MaxDate							MQ  ON MQ.ExposureKey = Q.ExposureKey AND MQ.ReferenceDate = Q.ReferenceDate
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Channels A	  ON A.ExposureKey = MQ.ExposureKey AND A.ReferenceDate = MQ.ReferenceDate
LEFT JOIN SavedFile									svf ON svf.Account = Q.ExposureKey AND svf.SavedDatePreviousBusinessDay <= (SELECT Distinct CalendarDate from referenceDateForHistorical)
LEFT JOIN queuesmapping	QMAP	ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription) 
LEFT JOIN assignmentlevelsmapping lv   ON LOWER(lv.`Full name 1`) = LOWER(A.AssignmentAgent) AND LOWER(lv.`1st level Assignment`) = LOWER(A.AssignmentChannel)
WHERE (svf.Account IS NULL OR svf.SavedDatePreviousBusinessDay > (SELECT Distinct CalendarDate from referenceDateForHistorical))
		
UNION ALL
		
SELECT DISTINCT     Q.ExposureKey
				,	Q.ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	A.AssignmentChannel	
				,	A.AssignmentTeam
				,	A.AssignmentAgent
				,	lv.`3rd level Assignment` AS AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	svf.SavedFlag
				,	svf.SavedReason 
				,	svf.SavedDate
FROM `{env_prefix}-cdh-delivery`.repoint.int_Queues	Q
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Channels A	  ON A.ExposureKey = Q.ExposureKey AND A.ReferenceDate = Q.ReferenceDate
INNER JOIN SavedFile									svf ON svf.Account = Q.ExposureKey AND svf.SavedDatePreviousBusinessDay = Q.ReferenceDate
LEFT JOIN queuesmapping	QMAP	ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription)
LEFT JOIN assignmentlevelsmapping lv   ON LOWER(lv.`Full name 1`) = LOWER(A.AssignmentAgent) AND LOWER(lv.`1st level Assignment`) = LOWER(A.AssignmentChannel)
WHERE svf.SavedDatePreviousBusinessDay <= (SELECT Distinct CalendarDate from referenceDateForHistorical)
''')
QueuesChannels.createOrReplaceTempView("QueuesChannels")   

In [0]:
AttrQueues_MaxDate = spark.sql(
f'''		
SELECT  MAX(allQ.ReferenceDate) AS ReferenceDate
	 ,  allQ.ExposureKey
FROM `{env_prefix}-cdh-delivery`.repoint.int_Queues allQ 
INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_AssignedPerimeter acc	ON ACC.ExposureKey = allQ.ExposureKey AND allQ.ReferenceDate < IFNULL(ACC.MtDClosingDate, '9999-01-01')
LEFT JOIN SavedFile svf	ON svf.Account = allQ.ExposureKey AND svf.SavedDatePreviousBusinessDay <= (SELECT Distinct CalendarDate from referenceDateForHistorical)
WHERE 1=1 AND svf.Account IS NULL
GROUP BY allQ.ExposureKey
''')
AttrQueues_MaxDate.createOrReplaceTempView("AttrQueues_MaxDate")     

##### Report Unified CL, ML, SB

In [0]:
extreport_unified_cl_som = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl
WHERE BusinessDate = {som_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl WHERE BusinessDate = {som_date!r})
''')
extreport_unified_cl_som.createOrReplaceTempView("extreport_unified_cl_som")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_cl_som.count() == 0:
    print(f"The required external file extreport_unified_cl dataset for date {som_date!r} is empty")

extreport_unified_ml_som = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml
WHERE BusinessDate = {som_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml WHERE BusinessDate = {som_date!r})
''')
extreport_unified_ml_som.createOrReplaceTempView("extreport_unified_ml_som")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_ml_som.count() == 0:
    print(f"The required external file extreport_unified_ml dataset for date {som_date!r} is empty")

extreport_unified_sb_som = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb
WHERE BusinessDate = {som_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb WHERE BusinessDate = {som_date!r})
''')
extreport_unified_sb_som.createOrReplaceTempView("extreport_unified_sb_som")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_sb_som.count() == 0:
    print(f"The required external file extreport_unified_sb dataset for date {som_date!r} is empty")


extreport_unified_cl = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl
WHERE BusinessDate = {run_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl WHERE BusinessDate = {run_date!r})
''')
extreport_unified_cl.createOrReplaceTempView("extreport_unified_cl")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_cl.count() == 0:
    print(f"The required external file extreport_unified_cl dataset for date {run_date!r} is empty")

extreport_unified_ml = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml
WHERE BusinessDate = {run_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml WHERE BusinessDate = {run_date!r})
''')
extreport_unified_ml.createOrReplaceTempView("extreport_unified_ml")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_ml.count() == 0:
    print(f"The required external file extreport_unified_ml dataset for date {run_date!r} is empty")

extreport_unified_sb = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb
WHERE BusinessDate = {run_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb WHERE BusinessDate = {run_date!r})
''')
extreport_unified_sb.createOrReplaceTempView("extreport_unified_sb")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_sb.count() == 0:
    print(f"The required external file extreport_unified_sb dataset for date {run_date!r} is empty")



In [0]:
Report_Unified_SoM = spark.sql(
f'''	
SELECT b.* 
FROM (
		SELECT * , RANK() OVER (PARTITION BY ExposureKey ORDER BY ApplicationCurrentStepInsertDate DESC, FirstDateAtValidation DESC) Rank
		FROM (
        SELECT 'CL'										AS BU
              ,CL.ExposureKey  
			  ,ApplicationNumber
			  ,CAST(CONCAT('20', left(ApplicationNumber,2), '-' , SUBSTRING(ApplicationNumber,3,2), '-' , SUBSTRING(ApplicationNumber,5,2)) AS DATE) AS ApplicationInsertDate
			  ,UnifiedApplicationStatus				AS ApplicationStatus
              ,CAST((CASE WHEN FirstDateReceiptValuation LIKE '%/%' THEN date_format(to_date(LEFT(FirstDateReceiptValuation,10), 'dd/MM/yyyy'),'yyyy-MM-dd') 
                    WHEN FirstDateReceiptValuation = ' ' THEN '1900-01-01'
                    ELSE LEFT(FirstDateReceiptValuation,10) END) AS DATE) AS FirstDateAtValidation
			  ,cast( CASE WHEN ReceiptDateCurrentStep LIKE '%/%' THEN date_format(to_date(LEFT(ReceiptDateCurrentStep,10), 'dd/MM/yyyy'),'yyyy-MM-dd') 
                    WHEN ReceiptDateCurrentStep = ' ' THEN '1900-01-01'
                    ELSE date_format(LEFT(ReceiptDateCurrentStep,10),'yyyy-MM-dd')  END as date) AS ApplicationCurrentStepInsertDate
        FROM extreport_unified_cl     CL
        WHERE ApplicationNumber <> ''  
          AND  BusinessDate = {som_date!r}
		      AND UnifiedApplicationStatus = 'Ενεργή'
		      AND FirstDateReceiptValuation IS NOT NULL
        
		    UNION ALL
        
        SELECT 'ML'										AS BU
              ,ML.ExposureKey  
              ,ApplicationNumber						AS ApplicationNumber
			  ,CAST(CONCAT('20', left(ApplicationNumber,2), '-' , SUBSTRING(ApplicationNumber,3,2), '-' , SUBSTRING(ApplicationNumber,5,2)) AS DATE) AS ApplicationInsertDate
			  ,UnifiedApplicationStatus								AS ApplicationStatus
			  ,CAST((CASE WHEN FirstDateReceiptValuation LIKE '%/%' THEN date_format(to_date(LEFT(FirstDateReceiptValuation,10), 'dd/MM/yyyy'),'yyyy-MM-dd') 
                    WHEN FirstDateReceiptValuation = ' ' THEN '1900-01-01'
                    ELSE LEFT(FirstDateReceiptValuation,10) END) AS DATE) AS FirstDateAtValidation
			  ,cast((CASE WHEN CurrentStepDate LIKE '%/%' THEN date_format(to_date(LEFT(CurrentStepDate,10), 'dd/MM/yyyy'),'yyyy-MM-dd') 
                    WHEN CurrentStepDate = ' ' THEN '1900-01-01'
                    ELSE date_format(LEFT(CurrentStepDate,10),'yyyy-MM-dd')  END) as date) AS ApplicationCurrentStepInsertDate
        FROM extreport_unified_ml   ML
        WHERE ApplicationNumber <> '' 
         AND BusinessDate = {som_date!r}
		     AND UnifiedApplicationStatus = 'Ενεργή'
		     AND FirstDateReceiptValuation IS NOT NULL

		UNION ALL
        
        SELECT 'SB'												AS BU
              ,SB.ExposureKey
              ,ApplicationNumber
			  ,CAST(CONCAT('20', left(ApplicationNumber,2), '-' , SUBSTRING(ApplicationNumber,3,2), '-' , SUBSTRING(ApplicationNumber,5,2)) AS DATE) AS ApplicationInsertDate
			  ,UnifiedApplicationStatus								AS ApplicationStatus
			  ,CAST((CASE WHEN FirstDateReceiptValuation LIKE '%/%' THEN date_format(to_date(LEFT(FirstDateReceiptValuation,10), 'dd/MM/yyyy'),'yyyy-MM-dd') 
                    WHEN FirstDateReceiptValuation = ' ' THEN '1900-01-01'
                    ELSE LEFT(FirstDateReceiptValuation,10) END) AS DATE) AS FirstDateAtValidation
			  ,CASE WHEN CurrentStepReceiptDate LIKE '%/%' THEN date_format(to_date(LEFT(CurrentStepReceiptDate,10), 'dd/MM/yyyy'),'yyyy-MM-dd')
                    WHEN LEFT(CurrentStepReceiptDate,10) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$' THEN date_format(to_date(LEFT(CurrentStepReceiptDate,10), 'yyyy-MM-dd'),'yyyy-MM-dd')
                    ELSE date_format(to_date(LEFT(CurrentStepReceiptDate,10), 'yyyy-MM-dd'),'yyyy-MM-dd') END  AS ApplicationCurrentStepInsertDate
        FROM extreport_unified_sb SB
        WHERE ApplicationNumber <> '' 
          AND BusinessDate = {som_date!r}
		      AND UnifiedApplicationStatus = 'Ενεργή'
		      AND FirstDateReceiptValuation IS NOT NULL) a
) b
''')
Report_Unified_SoM.createOrReplaceTempView("Report_Unified_SoM")     

In [0]:
ReportUnified_temp = spark.sql(f"""
SELECT 'CL' as AssetSegment,
BusinessDate,
ExposureKey ,
CASE WHEN FirstDateReceiptValuation LIKE '%/%' THEN date_format(to_date(LEFT(FirstDateReceiptValuation,10), 'dd/MM/yyyy'),'yyyy-MM-dd') 
                    WHEN FirstDateReceiptValuation = ' ' THEN '1900-01-01'
                    ELSE LEFT(FirstDateReceiptValuation,10) END FirstDateReceiptValuation,
ApplicationNumber as ApplicationNumber,
CommonApplicationStep as CommonApplicationStep,
Channel as Channel,
RegistratorCode,
RegistratorName,
cast( CASE WHEN ReceiptDateCurrentStep LIKE '%/%' THEN date_format(to_date(LEFT(ReceiptDateCurrentStep,10), 'dd/MM/yyyy'),'yyyy-MM-dd') 
                    WHEN ReceiptDateCurrentStep = ' ' THEN '1900-01-01'
                    ELSE date_format(LEFT(ReceiptDateCurrentStep,10),'yyyy-MM-dd')  END as date) AS ApplicationCurrentStepInsertDate
FROM extreport_unified_cl
WHERE BusinessDate = {run_date!r}

UNION ALL
 
SELECT 'ML' as AssetSegment,
BusinessDate,
ExposureKey  ,
CASE WHEN FirstDateReceiptValuation LIKE '%/%' THEN date_format(to_date(LEFT(FirstDateReceiptValuation,10), 'dd/MM/yyyy'),'yyyy-MM-dd') 
                    WHEN FirstDateReceiptValuation = ' ' THEN '1900-01-01'
                    ELSE LEFT(FirstDateReceiptValuation,10) END FirstDateReceiptValuation,
ApplicationNumber as ApplicationNumber,
CommonApplicationStep as CommonApplicationStep,
Channel as Channel,
RegistratorCode,
RegistratorName,
cast((CASE WHEN CurrentStepDate LIKE '%/%' THEN date_format(to_date(LEFT(CurrentStepDate,10), 'dd/MM/yyyy'),'yyyy-MM-dd') 
                    WHEN CurrentStepDate = ' ' THEN '1900-01-01'
                    ELSE date_format(LEFT(CurrentStepDate,10),'yyyy-MM-dd')  END) as date) AS ApplicationCurrentStepInsertDate
FROM extreport_unified_ml
WHERE BusinessDate = {run_date!r}

UNION ALL
 
SELECT 'SB' as AssetSegment,
BusinessDate,
ExposureKey  ,
CASE WHEN FirstDateReceiptValuation LIKE '%/%' THEN date_format(to_date(LEFT(FirstDateReceiptValuation,10), 'dd/MM/yyyy'),'yyyy-MM-dd') 
                    WHEN FirstDateReceiptValuation = ' ' THEN '1900-01-01'
                    ELSE LEFT(FirstDateReceiptValuation,10) END FirstDateReceiptValuation,
ApplicationNumber as ApplicationNumber,
'CommonApplicationStep_SB' as CommonApplicationStep,
Channel as Channel,
RegistratorCode,
RegistratorName,
CASE WHEN CurrentStepReceiptDate LIKE '%/%' THEN date_format(to_date(LEFT(CurrentStepReceiptDate,10), 'dd/MM/yyyy'),'yyyy-MM-dd')
                    WHEN LEFT(CurrentStepReceiptDate,10) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$' THEN date_format(to_date(LEFT(CurrentStepReceiptDate,10), 'yyyy-MM-dd'),'yyyy-MM-dd')
                    ELSE date_format(to_date(LEFT(CurrentStepReceiptDate,10), 'yyyy-MM-dd'),'yyyy-MM-dd') END AS ApplicationCurrentStepInsertDate
FROM extreport_unified_sb
WHERE BusinessDate = {run_date!r}
""")
 
ReportUnified_temp.createOrReplaceTempView("ReportUnified_temp")

In [0]:
Report_Unified_step1 = spark.sql(
f'''	
SELECT         A.AssetSegment					    AS BU
			  ,A.ExposureKey                       
              ,A.ApplicationNumber
			  ,A.CommonApplicationStep              AS ApplicationCurrentStep
              ,A.ApplicationCurrentStepInsertDate
              ,A.FirstDateReceiptValuation			AS FirstDateAtValidation			  
			  ,A.RegistratorCode					AS ApplicationRegistratorAM
			  ,A.RegistratorName 					AS ApplicationRegistratorName
			  ,L.ModificationDate
			  ,CASE WHEN A.Channel IN ('ALPHA BANK ΥΠΗΡΕΣΙΕΣ ΕΝΗΜΕΡΩΣΕΩΣ ΟΦΕΙΛΕΤΩΝ','CALL CENTER - ALPHA BANK ΥΠΗΡΕΣΙΕΣ ΕΝΗΜΕΡΩΣΕΩΣ','ΔΔΚΛΤ II',
											'ΔΔΟΛΤ - ΑΙΤΗΣΕΙΣ ΡΥΘΜΙΣΕΩΝ - Mitigating Actions','ΔΔΟΛΤ - ΑΙΤΗΣΕΙΣ ΡΥΘΜΙΣΕΩΝ - ΔΕΚ','ΔΔΟΛΤ - ΑΙΤΗΣΕΙΣ ΡΥΘΜΙΣΕΩΝ - ΕΙΔΙΚΟΥ ΧΕΙΡΙΣΜΟΥ') THEN 'IRT'
					WHEN A.Channel LIKE '%Δ.Γ.%' THEN 'Legal Agencies'
					WHEN A.Channel LIKE '%ΚΕΝΤΡΟ%' THEN 'HUB RMs'
					WHEN A.Channel LIKE '%CEPAL - INHOUSE%' OR A.Channel LIKE '%ΚΑΤΑΣΤΗΜΑ%' THEN NULL
				END Channel
        FROM ReportUnified_temp A
		LEFT JOIN (
		            SELECT OldExposureKey
					     , MIN(ModificationDate) AS ModificationDate 
					FROM Modifications 
					GROUP BY OldExposureKey
					) L   
		ON A.ExposureKey = L.OldExposureKey
        WHERE A.ApplicationNumber <> '' 
''')
Report_Unified_step1.createOrReplaceTempView("Report_Unified_step1")           

In [0]:
# %python
# create_view_sql = f"""
# CREATE OR REPLACE VIEW `qa-cdh-delivery`.repoint.core_applicationstepmapping
# WITH SCHEMA COMPENSATION
# AS
# SELECT ' ΔΚΛΤ - Αναμονή Ενεργειών' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Factoring - Εισήγηση' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'Leasing - Εισήγηση' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'NSRM - Απόφαση Senior Officer' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'NSRM - Απόφαση Διευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'NSRM - Απόφαση Υποδιευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'NSRM - Εισήγηση' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'NSRM - Επανεξέταση Απόφασης Senior Officer' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'NSRM - Επανεξέταση Απόφασης Διευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'NSRM - Επανεξέταση Απόφασης Υποδιευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'NSRM - Επανεξέταση Εισήγησης' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'Ακυρωμένη' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Ακυρωμένη Αίτηση' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Αναμονή Ενέργειας Καταστήματος' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Αναμονή Ενεργειών Καταστήματος' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Αξιολόγηση Αιτήματος ΔΔΟΛΤ' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'Αξιολόγηση Αιτήματος ΔΥΠ' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'Αξιολογηση αιτηματος-ΔΔΟΛΤ' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'Απόφαση - Ενημέρωση Καταστήματος' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση CO' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Senior CO' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Αίτησης/Αλλαγή Εγκριτικής Ευχέρειας' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Αίτησης/Απόφαση Officer A' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Αίτησης/Απόφαση Officer B' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Αίτησης/Απόφαση Senior Officer' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Αίτησης/Απόφαση για Αίτηση εκτός Ευχέρειας' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Αίτησης/Απόφαση Διευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Επιστροφή από Έλεγχο αιτήματος προς υλοποίηση' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Επιστροφή από Έλεγχο και Παραγωγή Συμβατικών Εγγράφων' AS CurrentStep, 'Επιστροφή μετά την Έγκριση' AS Category UNION
# SELECT 'ΔΚΛΤ - Επιστροφή από Υλοποίηση αιτήματος' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Εσφαλμένη Καταχώρηση' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Θεώρηση αίτησης' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Καταχώρηση αίτησης' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Παραλαβή Απόφασης' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'ΔΚΛΤ - Παραλαβή Απόφασης (Channel)' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'ΔΚΛΤ - Υπογραφή Καταστήματος' AS CurrentStep, 'Υπογραφή συμβάσεων από Κατάστημα/ΠΚ' AS Category UNION
# SELECT 'ΔΚΛΤ - Υπογραφή Συμβατικών Εγγράφων - Αίτημα με δι' AS CurrentStep, 'Υπογραφή συμβάσεων από Πελάτη και Κατάστημα/ΠΚ' AS Category UNION
# SELECT 'ΔΚΛΤ - Υπογραφή Συμβατικών Εγγράφων - Αίτημα με διαφορετικούς όρους' AS CurrentStep, 'Υπογραφή συμβάσεων από Πελάτη και Κατάστημα/ΠΚ' AS Category UNION
# SELECT 'ΔΚΛΤ-Αναμονη ενεργειων' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ-Επιστροφη από αξιολογηση' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ-Επιστροφη από αξιολογηση-Επανεξεταση' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΠΚ - Απόφαση Senior Officer' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΠΚ - Απόφαση Διευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΠΚ - Απόφαση Υποδιευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΠΚ - Επανεξέταση Απόφασης Senior Officer' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΠΚ - Επανεξέταση Απόφασης Διευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΠΚ - Επανεξέταση Απόφασης Υποδιευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΥΠ - Αναμονή Ενέργειών' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'ΔΥΠ - Δημιουργία Συμβατικών Εγγράφων' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'ΔΥΠ - Εγγραφή προσημειώσεως / Λήψη καλύμματος' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'ΔΥΠ - Εγγραφή Προσημείωσης/Λήψη καλύμματος' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'ΔΥΠ - Έλεγχος αιτήματος προς υλοποίηση' AS CurrentStep, 'ΔΥΠ προς υλοποίηση' AS Category UNION
# SELECT 'ΔΥΠ - Έλεγχος και Παραγωγή Συμβατικών Εγγράφων' AS CurrentStep, 'ΔΥΠ για δημιουργία συμβατικών εγγράφων' AS Category UNION
# SELECT 'ΔΥΠ - Θεώρηση Ελέγχου αιτήματος προς υλοποίηση' AS CurrentStep, 'ΔΥΠ προς υλοποίηση' AS Category UNION
# SELECT 'ΔΥΠ - Θεώρηση Ελέγχου και Παραγωγής Συμβατικών Εγγ' AS CurrentStep, 'ΔΥΠ για δημιουργία συμβατικών εγγράφων' AS Category UNION
# SELECT 'ΔΥΠ - Θεώρηση Ελέγχου και Παραγωγής Συμβατικών Εγγράφων' AS CurrentStep, 'ΔΥΠ για δημιουργία συμβατικών εγγράφων' AS Category UNION
# SELECT 'ΔΥΠ - Θεώρηση Υλοποίησης Αιτήματος' AS CurrentStep, 'ΔΥΠ προς υλοποίηση' AS Category UNION
# SELECT 'ΔΥΠ - Υλοποίηση Αιτήματος' AS CurrentStep, 'ΔΥΠ προς υλοποίηση' AS Category UNION
# SELECT 'ΔΥΠΛΤ / Ανάθεση Αιτήματος Εξασφαλίσεων σε Δικηγόρο' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΥΠΛΤ / Αναμονή Ν/Τ Ελέγχου' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'ΔΥΠΛΤ / Επεξεργασία Αίτησης Τμήματος Δανείων και Συμβάσεων' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'ΔΥΠ-Μ.Υ.Στ.Ε.Κα.Δ-Ολοκλήρωση Ενεργειών Καταστήματος' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Ενημερωση ολοκληρωσης ενεργειων' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Ενημέρωση Ολοκλήρωσης Ενεργειών' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Επανεξέταση Ρυθμίσεων από ΔΔΟΛΤ' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'Επικαιροποίηση Ρύθμισης από ΔΔΟΛΤ' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'Επιστροφη από αξιολογηση' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Κατάστημα - Αρχική Υπογραφή Συμβατικών Εγγράφων' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Κατάστημα - Εκκρεμότητα Εγγραφής Προσημείωσης' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Κατάστημα - Εκκρεμότητα Εξωτερικού Συνεργάτη' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'Κατάστημα - Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Κατάστημα - Επιστροφή από Αξιολόγηση' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Κατάστημα - Επιστροφή από Αξιολόγηση/Επανεξέταση' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Κατάστημα - Επιστροφή από ΔΥΠ' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Κατάστημα - Επιστροφή από Έλεγχο αιτήματος προς υλοποίηση' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'Κατάστημα - Επιστροφή από Έλεγχο και Παραγωγή Συμβατικών Εγγράφων' AS CurrentStep, 'Επιστροφή μετά την Έγκριση' AS Category UNION
# SELECT 'Κατάστημα - Επιστροφή από υλοποίηση' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'Κατάστημα - Επιστροφή από Υλοποίηση αιτήματος' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'Κατάστημα - Εσφαλμένη Καταχώρηση' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Κατάστημα - Θεώρηση αίτησης' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Κατάστημα - Καταχώρηση αίτησης' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Κατάστημα - Παραλαβή Απόφασης' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'Κατάστημα - Παραλαβή Απόφασης Εξωτερικών Συνεργατών' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'Κατάστημα - Υπογραφή Καταστήματος' AS CurrentStep, 'Υπογραφή συμβάσεων από Κατάστημα/ΠΚ' AS Category UNION
# SELECT 'Κατάστημα - Υπογραφή Συμβατικών Εγγράφων' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'Κατάστημα - Υπογραφή Συμβατικών Εγγράφων - Σύμβαση' AS CurrentStep, 'Υπογραφή συμβάσεων από Πελάτη και Κατάστημα/ΠΚ' AS Category UNION
# SELECT 'Κατάστημα - Υπογραφή Συμβατικών Εγγράφων - Σύμβαση με διαφορετικούς όρους' AS CurrentStep, 'Υπογραφή συμβάσεων από Πελάτη και Κατάστημα/ΠΚ' AS Category UNION
# SELECT 'Κατάστημα / Αναμονή Υπογραφής Συμβάσεων' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Κατάστημα / Ενημέρωση Ολοκλήρωσης Ενεργειών' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Κατάστημα / Επιστροφή από ΔΥΠΛΤ - Τμήμα Δανείων και Συμβάσεων' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'Κατάστημα / Επιστροφή από ΔΥΠΛΤ Έλεγχο Εξασφαλίσεων' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'Κατάστημα -Εκκρεμότητα Εξωτερικού Συνεργάτη' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'Κατάστημα -Ενημέρωση Ολοκλήρωση Διαδικασίας' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Καταστημα/Επιστροφη από ΔΥΠΛΤ-Τμημα Δανειων και Συμβασεων' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'Καταστημα-Καταχωρηση αιτησης' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'Παραλαβη αποφασεως' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'Παραλαβη αποφασης' AS CurrentStep, 'Παραλαβή Απόφασης' AS Category UNION
# SELECT 'Υλοποίηση Ρυθμίσεων' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'Υπογραφή Καταστήματος' AS CurrentStep, 'Υπογραφή συμβάσεων από Κατάστημα/ΠΚ' AS Category UNION
# SELECT 'Υπογραφή Συμβατικών Εγγράφων - Αίτημα με διαφορετικούς όρους' AS CurrentStep, 'Υπογραφή συμβάσεων από Πελάτη και Κατάστημα/ΠΚ' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Αίτησης/Απόφαση Συμβουλίου' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Αίτησης/Απόφαση Υποδιευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Αίτησης/Προετοιμασία Απόφασης' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Αξιολόγηση Υποδιευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Εκκρεμότητα Εξωτερικού Συνεργάτη' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Έλεγχος Αίτησης' AS CurrentStep, 'Ελεγχος Αίτησης- Validator' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επανεξέταση Αποφάσεως CO' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επανεξέταση Αποφάσεως SCO' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επανεξέταση Αποφάσεως Διευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επανεξέταση Αποφάσεως Υποδιευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επανεξέταση Απόφασης Decision Maker' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επανεξέταση Απόφασης Processor' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επανεξέταση Απόφασης Validator' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επανεξέταση Απόφασης Αίτησης εκτός Ευχέρειας' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επιστροφή Από Decision Maker' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επιστροφή από Έλεγχο αιτήματος προς υλοποί' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επιστροφή από Έλεγχο αιτήματος προς υλοποίηση' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επιστροφή από Έλεγχο και Παραγωγή Συμβατικ' AS CurrentStep, 'Επιστροφή μετά την Έγκριση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επιστροφή από Έλεγχο και Παραγωγή Συμβατικών Εγγράφων' AS CurrentStep, 'Επιστροφή μετά την Έγκριση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επιστροφή από υλοποίηση' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Επιστροφή από Υλοποίηση αιτήματος' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Θεώρηση Απόφασης Διευθυντή' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Προώθηση Απόφασης' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ - Προώθηση Απόφασης AC' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΔΟΛΤ-Αξιολογηση' AS CurrentStep, 'ΔΔΟΛΤ-Αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Αναμονή Ενεργειών' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Αρχική Υπογραφή Συμβατικών Εγγράφων' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Εκκρεμότητα Εγγραφής Προσημείωσης' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'ΔΚΛΤ - Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category UNION
# SELECT 'ΔΚΛΤ - Επιστροφή από  υλοποίηση' AS CurrentStep, 'Επιστροφή από την υλοποίηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Επιστροφή από Αξιολόγηση' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Επιστροφή από Αξιολόγηση (Cepal)' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Επιστροφή από Αξιολόγηση (Channel)' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Επιστροφή από Αξιολόγηση (Εξωτ. Συνεργάτες)' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Επιστροφή από Αξιολόγηση/Επανεξέταση' AS CurrentStep, 'Επιστροφή από αξιολόγηση' AS Category UNION
# SELECT 'ΔΚΛΤ - Επιστροφή από ΔΥΠ' AS CurrentStep, 'Ενημέρωση Ολοκλήρωσης Διαδικασίας' AS Category 

# """
# spark.sql(create_view_sql)

In [0]:
Report_Unified = spark.sql(
f'''			
SELECT	* 
	  , step.Category AS StepCategory
FROM (SELECT * 
		   , RANK() OVER(PARTITION BY ExposureKey ORDER BY FirstDateAtValidation DESC, ApplicationCurrentStepInsertDate DESC) AS Rank
	  FROM Report_Unified_step1)  A
	  LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.core_applicationstepmapping  step  ON step.CurrentStep = A.ApplicationCurrentStep  
WHERE A.Rank = 1
''')
Report_Unified.createOrReplaceTempView("Report_Unified")   

##### SavedFile_Mods

In [0]:
SavedFile_Mods = spark.sql(
f'''		
SELECT s.*
	 , u.ApplicationRegistratorAM
	 , u.ApplicationRegistratorName
	 , u.Channel
FROM SavedFile s
LEFT JOIN Modifications m ON m.OldExposureKey = s.Account 
LEFT JOIN Report_Unified u ON s.Account = u.ExposureKey
WHERE SavedReason IN ('Ρύθμιση Haircut','Συμβατική Ρύθμιση','Ρυθμίσεις N3869')
AND IFNULL(m.ModificationDate,'1900-01-01') > {som_date!r} 
AND IFNULL(m.ModificationDate,'1900-01-01') <= {run_date!r}
''')
SavedFile_Mods.createOrReplaceTempView("SavedFile_Mods") 

##### Unified_Apps_Impls

In [0]:
Unified_Apps = spark.sql(
f'''		
SELECT u.*
FROM Report_Unified u
LEFT JOIN Modifications m ON m.OldExposureKey = u.ExposureKey AND m.ModificationDate = u.ModificationDate 
LEFT JOIN SavedFile s ON s.Account = u.ExposureKey AND s.SavedReason IN ('Ρύθμιση Haircut','Συμβατική Ρύθμιση')
LEFT JOIN Report_Unified_SoM u_SoM ON u_SoM.ExposureKey = u.ExposureKey AND u_SoM.ApplicationNumber = u.ApplicationNumber 
WHERE s.Account IS NULL
AND m.OldExposureKey IS NULL
AND u.FirstDateAtValidation IS NOT NULL 
AND (u.FirstDateAtValidation > {som_date!r} AND u.FirstDateAtValidation <= {run_date!r}) 
''')
Unified_Apps.createOrReplaceTempView("Unified_Apps")     

In [0]:
Unified_Impls = spark.sql(
f'''		
SELECT u.*
FROM Report_Unified u
INNER JOIN Modifications m ON m.OldExposureKey = u.ExposureKey AND m.ModificationDate = u.ModificationDate 
LEFT JOIN SavedFile s ON s.Account = u.ExposureKey AND s.SavedReason IN ('Ρύθμιση Haircut','Συμβατική Ρύθμιση')
WHERE s.Account IS NULL
AND (IFNULL(m.ModificationDate,'1900-01-01') > {som_date!r} 
AND IFNULL(m.ModificationDate,'1900-01-01') <= {run_date!r})
AND Categoryfinal <> 'Extension'
''')
Unified_Impls.createOrReplaceTempView("Unified_Impls")     

In [0]:
Unified_Apps_Impls = spark.sql(
f'''		
SELECT * 
FROM Unified_Apps
UNION ALL
SELECT * 
FROM Unified_Impls
''')
Unified_Apps_Impls.createOrReplaceTempView("Unified_Apps_Impls")   

##### Activities

In [0]:
Activities = spark.sql(
f'''			
SELECT *, RANK() OVER(PARTITION BY ExposureKey ORDER BY InteractionDate DESC, InteractionTypeKey ASC, CASE WHEN PartyRelationshipType = 'Primary Owner' THEN 1 ELSE 0 END DESC, CreatedDate DESC) AS RANK 
FROM (
		SELECT DISTINCT 
					A.ExposureKey 
				,	A.InteractionTypeKey 
				,	A.InteractionType 
				,	A.InteractionDate 
				,	B.EmployeeFullName AS CreatedBy		
				,	B.EmployeeUserName AS CreatedById
				,	nc.PartyRelationshipType
				,	A.CreatedDate
		FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) A
		INNER JOIN `{env_prefix}-cdh`.primary.exposure nc on A.ExposureKey = nc.ExposureKey and nc.BusinessDate={run_date!r}
        LEFT JOIN `{env_prefix}-cdh`.primary.firmstructure B ON A.CreatedBy = B.EmployeePartyKey AND B.EmployeeCategory = "Employee" AND B.SourceSystem = 'qca' AND B.BusinessDate={run_date!r}
		WHERE A.InteractionTypeKey IN ('620','486')
          AND A.PortfolioName = 'Core Retail'
		  AND A.InteractionDate <= {run_date!r}
		  AND A.InteractionDate > {som_date!r}
)
''')
Activities.createOrReplaceTempView("Activities")     

In [0]:
LatestActivities_ext = spark.sql(
f'''	
SELECT * 
FROM Activities
WHERE Rank = 1
''')
LatestActivities_ext.createOrReplaceTempView("LatestActivities_ext")  

In [0]:
SavedFile_Ext = spark.sql(
f'''			
SELECT s.*
	 , u.CreatedBy
	 , u.CreatedById
FROM SavedFile                  AS s
INNER JOIN LatestActivities_ext AS u ON s.Account = u.ExposureKey AND SavedReason IN ('Extension')
''')
SavedFile_Ext.createOrReplaceTempView("SavedFile_Ext")    

In [0]:
Unified_Apps_Ext = spark.sql(
f'''			
SELECT m.* , e.CreatedBy, e.CreatedById
FROM Modifications m
LEFT JOIN SavedFile s ON s.Account = m.OldExposureKey AND SavedReason IN ('Extension') 
LEFT JOIN LatestActivities_ext e ON e.ExposureKey = m.OldExposureKey 
WHERE s.Account IS NULL
	AND ((IFNULL(m.ModificationDate,'1900-01-01') > {som_date!r} AND IFNULL(m.ModificationDate,'1900-01-01') <= {run_date!r}))
	AND m.Categoryfinal = 'Extension'
''')
Unified_Apps_Ext.createOrReplaceTempView("Unified_Apps_Ext")     

In [0]:
SavedFile_Pay = spark.sql(
f'''		
SELECT *
FROM SavedFile 
WHERE SavedReason IN ('Payments')
''')
SavedFile_Pay.createOrReplaceTempView("SavedFile_Pay")   

In [0]:
SavedFile_Rest = spark.sql(
f'''			
SELECT *
FROM SavedFile 
WHERE SavedReason IN ('Other Reason')
''')
SavedFile_Rest.createOrReplaceTempView("SavedFile_Rest")  

##### Payments

In [0]:
Payments = spark.sql(
f'''			
SELECT DISTINCT 
				p.ExposureKey 
			,	p.PaymentActualDate 
			,	CASE WHEN c.IsBusinessDay = 0 THEN c.NextBusinessDay ELSE p.PaymentActualDate END AS PaymentNextBusinessDate
			,	PreviousBusinessDay AS PaymentPreviousBusinessDate
			,	SUM(PaymentAmount) AS PaymentAmount
FROM `{env_prefix}-cdh`.primary.payment p
INNER JOIN `{env_prefix}-cdh`.reference.ref_calendar c	ON c.CalendarDate  = p.PaymentActualDate
WHERE BusinessDate = LAST_DAY({run_date!r})
AND p.PaymentActualDate <= {run_date!r}
AND p.PaymentActualDate > {som_date!r}
GROUP BY ExposureKey
	    ,p.PaymentActualDate
		,CASE WHEN c.IsBusinessDay = 0 THEN c.NextBusinessDay ELSE p.PaymentActualDate END
		,PreviousBusinessDay
''')
Payments.createOrReplaceTempView("Payments")      

##### KeptPromise

In [0]:
KeptPromise = spark.sql(
f'''	
SELECT DISTINCT 
					A.ExposureKey 
				,	A.InteractionDate 
				,	B.EmployeeFullName AS CreatedBy		
				,	B.EmployeeUserName AS CreatedById
FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) A
LEFT JOIN `{env_prefix}-cdh`.primary.firmstructure B ON A.CreatedBy = B.EmployeePartyKey AND B.EmployeeCategory = "Employee" AND B.SourceSystem = 'qca' AND B.BusinessDate={run_date!r}
-- INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_AssignedPerimeter C ON C.ExposureKey = A.ExposureKey
INNER JOIN Payments          D ON D.ExposureKey = A.ExposureKey 
WHERE A.PortfolioName = 'Core Retail'
  AND A.InteractionDate <= {run_date!r} 
  AND A.InteractionDate > {som_date!r} 
  AND A.InteractionCategory IN ('PROMISE','ΥΠΟΣΧΕΣΗ')
''')
KeptPromise.createOrReplaceTempView("KeptPromise")   

##### AttrQueuesChannels

In [0]:
AttrQueuesChannels = spark.sql(
f'''	
SELECT DISTINCT 	Q.ExposureKey
				,	Q.ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	A.AssignmentChannel	
				,	A.AssignmentTeam
				,	A.AssignmentAgent
				,	lv.`3rd level Assignment` AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	svf.SavedFlag
				,	svf.SavedReason 
				,	svf.SavedDate
				,	'w/o Event' Event
FROM `{env_prefix}-cdh-delivery`.repoint.int_Queues Q		
INNER JOIN AttrQueues_MaxDate  MQ	ON MQ.ExposureKey = Q.ExposureKey AND MQ.ReferenceDate = Q.ReferenceDate
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Channels A	ON A.ExposureKey = MQ.ExposureKey AND A.ReferenceDate = MQ.ReferenceDate
LEFT JOIN SavedFile	 svf ON svf.Account = Q.ExposureKey AND svf.SavedDatePreviousBusinessDay <= {run_date!r} 
LEFT JOIN Unified_Apps_Impls uf ON uf.ExposureKey = Q.ExposureKey 
LEFT JOIN Unified_Apps_Ext ext ON ext.OldExposureKey = Q.ExposureKey 
LEFT JOIN queuesmapping	QMAP ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription)
LEFT JOIN assignmentlevelsmapping lv	ON lower(lv.`Full name 1`) = lower(A.AssignmentAgent) AND lower(lv.`1st level Assignment`) = lower(A.AssignmentChannel)
LEFT JOIN Payments pm	ON pm.ExposureKey = Q.ExposureKey 
WHERE svf.Account IS NULL AND uf.ExposureKey IS NULL AND pm.ExposureKey IS NULL AND ext.OldExposureKey IS NULL
		
UNION ALL
		
SELECT DISTINCT     svf.Account
				,	Q.ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	svf.Channel AssignmentChannel	
				,	lv.`2nd level Assignment` AssignmentTeam
				,	svf.ApplicationRegistratorName AssignmentAgent
				,	svf.ApplicationRegistratorAM AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	svf.SavedFlag
				,	svf.SavedReason 
				,	svf.SavedDate
				,	'Saved due to Restructure' Event
FROM SavedFile_Mods	 svf
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Queues Q		ON svf.Account = Q.ExposureKey AND svf.SavedDatePreviousBusinessDay = Q.ReferenceDate
LEFT JOIN queuesmapping	QMAP	ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription)
LEFT JOIN assignmentlevelsmapping lv	ON 
lower(IF(LEN(TRIM(lv.`3rd level Assignment`))=6,RIGHT(lv.`3rd level Assignment`,5),lv.`3rd level Assignment`))=lower(IF(LEN(TRIM(svf.ApplicationRegistratorAM))=6,RIGHT(svf.ApplicationRegistratorAM,5),svf.ApplicationRegistratorAM))
WHERE svf.SavedDatePreviousBusinessDay <= {run_date!r} 
		
UNION ALL
		
SELECT DISTINCT     svf.Account
				,	Q.ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	lv.`1st level Assignment` AssignmentChannel	
				,	lv.`2nd level Assignment` AssignmentTeam
				,	svf.CreatedBy AssignmentAgent
				,	svf.CreatedById AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	svf.SavedFlag
				,	svf.SavedReason 
				,	svf.SavedDate
				,	'Saved due to Extension' Event
FROM SavedFile_Ext												svf
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Queues Q		ON svf.Account = Q.ExposureKey AND svf.SavedDate = Q.ReferenceDate
LEFT JOIN queuesmapping	QMAP	ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription)
LEFT JOIN assignmentlevelsmapping lv    ON LOWER(lv.`3rd level Assignment`) = LOWER(svf.CreatedById)
WHERE svf.SavedDatePreviousBusinessDay <= {run_date!r} 
		
UNION ALL
		
SELECT DISTINCT     svf.Account
				,	Q.ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	A.AssignmentChannel	
				,	A.AssignmentTeam
				,	A.AssignmentAgent
				,	lv.`3rd level Assignment` AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	svf.SavedFlag
				,	svf.SavedReason 
				,	svf.SavedDate
				,	'Saved due to Payment' Event
FROM SavedFile_Pay	svf
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Queues	Q		ON svf.Account = Q.ExposureKey AND svf.SavedDatePreviousBusinessDay = Q.ReferenceDate
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Channels	A		ON A.ExposureKey = Q.ExposureKey AND A.ReferenceDate = Q.ReferenceDate
LEFT JOIN queuesmapping	QMAP	ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription)
LEFT JOIN assignmentlevelsmapping lv	ON LOWER(lv.`Full name 1`) = LOWER(A.AssignmentAgent) AND LOWER(lv.`1st level Assignment`) = LOWER(A.AssignmentChannel)
WHERE svf.SavedDatePreviousBusinessDay <= {run_date!r} 
		
UNION ALL
		
SELECT DISTINCT     svf.Account
				,	Q.ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	A.AssignmentChannel	
				,	A.AssignmentTeam
				,	A.AssignmentAgent
				,	lv.`3rd level Assignment` AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	svf.SavedFlag
				,	svf.SavedReason 
				,	svf.SavedDate
				,	'Saved due to Other Reason' Event
FROM SavedFile_Rest	svf
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Queues Q	ON svf.Account = Q.ExposureKey AND svf.SavedDatePreviousBusinessDay = Q.ReferenceDate
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Channels	A		ON A.ExposureKey = Q.ExposureKey AND A.ReferenceDate = Q.ReferenceDate
LEFT JOIN queuesmapping	QMAP	ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription)
LEFT JOIN assignmentlevelsmapping lv	ON LOWER(lv.`Full name 1`) = LOWER(A.AssignmentAgent) AND LOWER(lv.`1st level Assignment`) = LOWER(A.AssignmentChannel)
WHERE svf.SavedDatePreviousBusinessDay <= {run_date!r} 
''')
AttrQueuesChannels.createOrReplaceTempView("AttrQueuesChannels")     

##### AttrQueuesChannels_Payments

In [0]:
AttrQueuesChannels_Payments = spark.sql(
f'''			
SELECT DISTINCT 
					ExposureKey
				,	AssignmentChannel	
				,	AssignmentTeam
				,	AssignmentAgent
				,	AssignmentAgentAM
				,	SUM(PaymentAmount) PaymentAmount
FROM (
		      SELECT
						P.ExposureKey
					,	Attr.Queues
					,	Attr.Agency
					,	Attr.AssignmentChannel	
					,	Attr.AssignmentTeam
					,	Attr.AssignmentAgent
					,	Attr.AssignmentAgentAM
					,	QMAP.CourseOfAction 
					,	QMAP.AssignmentSegmentDetailed
					,	P.PaymentAmount
		      FROM Payments							       P		
		      INNER JOIN AttrQueuesChannels	Attr ON Attr.ExposureKey = P.ExposureKey AND Attr.SavedDate>=P.PaymentActualDate  AND Attr.Event = 'Saved due to Payment'
		      LEFT JOIN queuesmapping QMAP ON LOWER(Attr.Queues) = LOWER(QMAP.SavedQueueDescription) 
			  LEFT JOIN assignmentlevelsmapping lv   ON LOWER(lv.`Full name 1`) = LOWER(Attr.AssignmentAgent)
			  WHERE IFNULL(Attr.AssignmentChannel,'-1') = 'HUB RMs'
		      
		      UNION ALL
		      
		      SELECT
					ExposureKey
				,	Queues
				,	Agency
				,	AssignmentChannel	
				,	AssignmentTeam
				,	AssignmentAgent
				,	AssignmentAgentAM
				,	CourseOfAction 
				,	AssignmentSegmentDetailed
				,	PaymentAmount
		      FROM (
		             SELECT
						        	P.ExposureKey
						        ,	Attr.Queues
						        ,	Attr.Agency
						        ,	Attr.AssignmentChannel	
						        ,	Attr.AssignmentTeam
						        ,	kept.CreatedById AssignmentAgent
						        ,	kept.CreatedById AssignmentAgentAM
						        ,	QMAP.CourseOfAction 
						        ,	QMAP.AssignmentSegmentDetailed
						        ,	P.PaymentAmount
						        ,	RANK() OVER(PARTITION BY P.ExposureKey, P.PaymentActualDate ORDER BY InteractionDate DESC) Rank
		             FROM Payments					           P		
		             INNER JOIN AttrQueuesChannels	Attr ON Attr.ExposureKey = P.ExposureKey AND Attr.SavedDate>=P.PaymentActualDate  AND Attr.Event = 'Saved due to Payment'
		             LEFT JOIN queuesmapping QMAP ON LOWER(Attr.Queues) = LOWER(QMAP.SavedQueueDescription)
		             LEFT JOIN KeptPromise  kept ON kept.ExposureKey = P.ExposureKey AND kept.InteractionDate <= P.PaymentActualDate
		             WHERE IFNULL(Attr.AssignmentChannel,'-1') <> 'HUB RMs'
		      	  ) A
		      WHERE A.Rank = 1
		) A
GROUP BY ExposureKey
			 , AssignmentChannel	
			 , AssignmentTeam
			 , AssignmentAgent
			 , AssignmentAgentAM
''')
AttrQueuesChannels_Payments.createOrReplaceTempView("AttrQueuesChannels_Payments")       

##### AttrQueuesChannels_Rest

In [0]:
AttrQueuesChannels_Rest = spark.sql(
f'''	
SELECT DISTINCT 
					ExposureKey
				,	MAX(ReferenceDate) ReferenceDate
				,	Queues
				,	Agency
				,	AssignmentChannel	
				,	AssignmentTeam
				,	AssignmentAgent
				,	AssignmentAgentAM
				,	CourseOfAction 
				,	AssignmentSegmentDetailed
				,	SUM(PaymentAmount) PaymentAmount
				,	Event
		FROM (		
		SELECT DISTINCT 
					P.ExposureKey
				,	P.PaymentPreviousBusinessDate ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	A.AssignmentChannel	
				,	A.AssignmentTeam
				,	A.AssignmentAgent
				,	lv.`3rd level Assignment` AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	P.PaymentAmount
				,	'Payment' Event
		FROM Payments P
		INNER JOIN AttrQueuesChannels Attr  ON Attr.ExposureKey = P.ExposureKey 
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Queues Q		ON P.ExposureKey = Q.ExposureKey AND P.PaymentPreviousBusinessDate = Q.ReferenceDate
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Channels A		ON A.ExposureKey = Q.ExposureKey AND A.ReferenceDate = Q.ReferenceDate
		LEFT JOIN queuesmapping QMAP ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription)
		INNER JOIN SavedFile_Pay svf	ON svf.Account = Attr.ExposureKey 
        LEFT JOIN assignmentlevelsmapping lv   ON LOWER(lv.`Full name 1`) = LOWER(A.AssignmentAgent) AND LOWER(lv.`1st level Assignment`) = LOWER(A.AssignmentChannel)
		WHERE (svf.Account IS NOT NULL AND svf.SavedDate < P.PaymentPreviousBusinessDate  AND A.AssignmentChannel = 'HUB RMs') 
		   		
		UNION ALL
		
		SELECT DISTINCT 
					P.ExposureKey
				,	P.PaymentPreviousBusinessDate ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	A.AssignmentChannel	
				,	A.AssignmentTeam
				,	A.AssignmentAgent
				,	lv.`3rd level Assignment` AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	P.PaymentAmount
				,	'Payment' Event
		FROM Payments P
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Queues Q		ON P.ExposureKey = Q.ExposureKey AND P.PaymentPreviousBusinessDate = Q.ReferenceDate
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Channels A		ON A.ExposureKey = Q.ExposureKey AND A.ReferenceDate = Q.ReferenceDate
		LEFT JOIN queuesmapping QMAP ON LOWER(Q.Queues) =LOWER(QMAP.SavedQueueDescription)
		LEFT JOIN SavedFile_Pay											svf	ON svf.Account = P.ExposureKey 
		LEFT JOIN assignmentlevelsmapping lv   ON LOWER(lv.`Full name 1`) = LOWER(A.AssignmentAgent) AND LOWER(lv.`1st level Assignment`) = LOWER(A.AssignmentChannel)
		WHERE svf.Account IS NULL AND IFNULL(A.AssignmentChannel,'-1') = 'HUB RMs' 
		
		UNION ALL
		
		SELECT DISTINCT 
					ExposureKey
				,	ReferenceDate
				,	Queues
				,	Agency
				,	AssignmentChannel	
				,	AssignmentTeam
				,	AssignmentAgent
				,	AssignmentAgentAM
				,	CourseOfAction 
				,	AssignmentSegmentDetailed
				,	PaymentAmount
				,	Event 
		FROM (
		SELECT DISTINCT 
					P.ExposureKey
				,	P.PaymentPreviousBusinessDate ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	A.AssignmentChannel	
				,	A.AssignmentTeam
				,	kept.CreatedBy AssignmentAgent
				,	kept.CreatedById AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	P.PaymentAmount
				,	'Payment' Event
				,	RANK() OVER(PARTITION BY P.ExposureKey, P.PaymentActualDate ORDER BY InteractionDate DESC) Rank
		FROM Payments P
		INNER JOIN AttrQueuesChannels Attr  ON Attr.ExposureKey = P.ExposureKey 
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Queues Q		ON P.ExposureKey = Q.ExposureKey AND P.PaymentPreviousBusinessDate = Q.ReferenceDate
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Channels A ON A.ExposureKey = Q.ExposureKey AND A.ReferenceDate = Q.ReferenceDate
		LEFT JOIN queuesmapping QMAP ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription)
		LEFT JOIN KeptPromise kept  ON kept.ExposureKey = P.ExposureKey AND kept.InteractionDate <= P.PaymentActualDate
		INNER JOIN SavedFile_Pay svf	ON svf.Account = Attr.ExposureKey 
		WHERE (svf.Account IS NOT NULL AND svf.SavedDate < P.PaymentPreviousBusinessDate AND IFNULL(A.AssignmentChannel,'-1') <> 'HUB RMs') 
		
		   ) A
		WHERE A.Rank = 1
		
		UNION ALL
		
		SELECT DISTINCT
					ExposureKey
				,	ReferenceDate
				,	Queues
				,	Agency
				,	AssignmentChannel	
				,	AssignmentTeam
				,	AssignmentAgent
				,	AssignmentAgentAM
				,	CourseOfAction 
				,	AssignmentSegmentDetailed
				,	PaymentAmount
				,	Event 
		FROM (
		SELECT DISTINCT 
					P.ExposureKey
				,	P.PaymentPreviousBusinessDate ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	A.AssignmentChannel	
				,	A.AssignmentTeam
				,	kept.CreatedBy AssignmentAgent
				,	kept.CreatedById AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	P.PaymentAmount
				,	'Payment' Event
				,	RANK() OVER(PARTITION BY P.ExposureKey, P.PaymentActualDate ORDER BY InteractionDate DESC) Rank
		FROM Payments P
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Queues Q		ON P.ExposureKey = Q.ExposureKey AND P.PaymentPreviousBusinessDate = Q.ReferenceDate
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Channels A ON A.ExposureKey = Q.ExposureKey AND A.ReferenceDate = Q.ReferenceDate
		LEFT JOIN queuesmapping QMAP ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription)
		LEFT JOIN KeptPromise kept  ON kept.ExposureKey = P.ExposureKey AND kept.InteractionDate <= P.PaymentActualDate
		LEFT JOIN SavedFile_Pay svf	ON svf.Account = P.ExposureKey 
		WHERE svf.Account IS NULL AND IFNULL(A.AssignmentChannel,'-1') <> 'HUB RMs') A
		WHERE A.Rank = 1
		) A
		GROUP BY ExposureKey,	Queues,	Agency,	AssignmentChannel, AssignmentTeam, AssignmentAgent,	AssignmentAgentAM , CourseOfAction, AssignmentSegmentDetailed, Event
		
		UNION ALL
		
		SELECT DISTINCT 
					uf.ExposureKey
				,	IFNULL(uf.ModificationDate,uf.FirstDateAtValidation) ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	uf.Channel AssignmentChannel	
				,	lv.`2nd level Assignment` AssignmentTeam
				,	IFNULL(uf.ApplicationRegistratorName,uf.ApplicationRegistratorName) AssignmentAgent
				,	IFNULL(uf.ApplicationRegistratorAM,uf.ApplicationRegistratorAM) AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	0 PaymentAmount
				,	CASE WHEN uf.ModificationDate IS NOT NULL THEN 'Restructure' ELSE 'Active Application' END Event
		FROM Unified_Apps_Impls uf
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Queues Q		ON uf.ExposureKey = Q.ExposureKey AND Q.ReferenceDate = IFNULL(uf.ModificationDate,uf.FirstDateAtValidation)
		LEFT JOIN queuesmapping QMAP ON LOWER(Q.Queues) = LOWER(QMAP.SavedQueueDescription)
		LEFT JOIN SavedFile_Mods svf	ON svf.Account = uf.ExposureKey AND svf.SavedDatePreviousBusinessDay <= {run_date!r}
		LEFT JOIN assignmentlevelsmapping lv   ON LOWER(lv.`3rd level Assignment`) = LOWER(uf.ApplicationRegistratorAM) AND LOWER(lv.`1st level Assignment`) = LOWER(uf.Channel)
		WHERE svf.Account IS NULL
		
		UNION ALL
		
		SELECT DISTINCT 
					uf.OldExposureKey
				,	uf.ModificationDate ReferenceDate
				,	Q.Queues
				,	Q.Agency
				,	lv.`1st level Assignment` AssignmentChannel	
				,	lv.`2nd level Assignment` AssignmentTeam
				,	lv.`Full name` AssignmentAgent
				,	uf.CreatedById AssignmentAgentAM
				,	QMAP.CourseOfAction 
				,	QMAP.AssignmentSegmentDetailed
				,	0 PaymentAmount
				,	CASE WHEN uf.ModificationDate IS NOT NULL THEN 'Extension' END Event
		FROM Unified_Apps_Ext uf
		LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_Queues Q		ON uf.OldExposureKey = Q.ExposureKey AND Q.ReferenceDate = ModificationDate
		LEFT JOIN queuesmapping QMAP ON Q.Queues = QMAP.SavedQueueDescription
		LEFT JOIN SavedFile_Mods svf	ON svf.Account = uf.OldExposureKey AND svf.SavedDate <= {run_date!r}
        LEFT JOIN assignmentlevelsmapping lv   ON LOWER(lv.`3rd level Assignment`) = LOWER(uf.CreatedById)
		WHERE svf.Account IS NULL
''')
AttrQueuesChannels_Rest.createOrReplaceTempView("AttrQueuesChannels_Rest")    

##### AssignedPerimeter_v1

In [0]:
AssignedPerimeter_v1_0 = spark.sql(
f'''	
SELECT DISTINCT     per.PortfolioName
				,	per.ExposureKey 
				,	per.AgreementKey 
				,	per.CDI 
				,	per.PartyTaxRegistrationNumber
				,	per.Entity
				,	per.AssetSegment
				,	per.ProductKey
				,	per.SoMBalanceAmountEUR
				,	per.SoMSourceBucket
				,	per.SoMObjectiveDate
				,	per.SoMObjectiveStatus
				, 	per.SoMDenouncedStatus
				,	per.MtDQueues
				,	per.MtDAgency
				,	UN.RiskBucket
				,	UN.EBAStatus
				,	eba.`PE/NPEDebtor`
				,	QCH.AssignmentChannel																								
				,	QCH.AssignmentTeam																									
				,	QCH.AssignmentAgent		
				,	QCH.AssignmentAgentAM																									
				,	QCH.Queues																												
				,	QCH.CourseOfAction																									
				,	QCH.AssignmentSegmentDetailed
				,	per.Anchor
				,	SV.SavedFlag
				,	SV.SavedReason 
				,	SV.SavedDate
				,	CASE WHEN SV.SavedFlag = 'Saved' THEN AttrQCH.Queues END     AS SavedQueues
				,	CASE WHEN SV.SavedFlag = 'Saved' THEN AttrQCH.Agency END AS SavedAgency
				,	d.Treatmentbranch
				,	e.SubProduct
				,	AttrQCH.CourseOfAction AttributionCOA
				,	AttrQCH.AssignmentChannel AttributionChannel
				,	AttrQCH.AssignmentTeam AttributionTeam
				,	CASE WHEN IFNULL(AttrQCH.AssignmentChannel,'-1') <> 'HUB RMs' AND AttrQCH.Event = 'Saved due to Payments' THEN  AttrQCHP.AssignmentAgent ELSE AttrQCH.AssignmentAgent END AttributionAgent
				,	CASE WHEN IFNULL(AttrQCH.AssignmentChannel,'-1') <> 'HUB RMs' AND AttrQCH.Event = 'Saved due to Payments' THEN  AttrQCHP.AssignmentAgentAM ELSE AttrQCH.AssignmentAgentAM END AttributionAgentAM
				,	CAST(NULL AS VARCHAR(100)) AttributionAgent_2ndLevel
				,	CAST(NULL AS VARCHAR(100)) AttributionAgentAM_2ndLevel
				,	AttrQCH.AssignmentSegmentDetailed AttributionSegmentDetailed
				,	AttrQCH.Event
				,	AttrQCHP.PaymentAmount
				,	TRIM(AttrQCH.AssignmentChannel) AttributionChannelOriginal
				,	TRIM(AttrQCH.AssignmentTeam) AttributionTeamOriginal
				,	TRIM(AttrQCH.AssignmentAgent) AttributionAgentOriginal
				,	TRIM(AttrQCH.AssignmentAgentAM) AttributionAgentAMOriginal
		FROM `{env_prefix}-cdh-delivery`.repoint.int_AssignedPerimeter						  AS per 
		LEFT JOIN UnionFile							  AS UN		ON per.ExposureKey = UN.ExposureNumber
		LEFT JOIN AFMTotalEBA						  AS eba		ON eba.Entity = per.Entity
		LEFT JOIN QueuesChannels						  AS QCH		ON per.ExposureKey = QCH.ExposureKey
		-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_DailyPortfolio AS D		ON D.ACCOUNT_NUMBER = per.ExposureKey
        LEFT JOIN Treatmentbranch D	 ON D.ACCOUNT_NUMBER = per.ExposureKey
        LEFT JOIN SubProduct E	 ON E.ACCOUNT_NUMBER = per.ExposureKey
		INNER JOIN AttrQueuesChannels				  AS AttrQCH  ON per.ExposureKey = AttrQCH.ExposureKey 
		LEFT JOIN AttrQueuesChannels_Payments		  AS AttrQCHP ON per.ExposureKey = AttrQCHP.ExposureKey
		LEFT JOIN SavedInfo							  AS SV		ON per.ExposureKey = SV.Account
		
		UNION ALL

		SELECT DISTINCT per.PortfolioName
				,	per.ExposureKey 
				,	per.AgreementKey 
				,	per.CDI 
				,	per.PartyTaxRegistrationNumber
				,	per.Entity
				,	per.AssetSegment
				,	per.ProductKey
				,	per.SoMBalanceAmountEUR
				,	per.SoMSourceBucket
				,	per.SoMObjectiveDate
				,	per.SoMObjectiveStatus
				, per.SoMDenouncedStatus
				,	per.MtDQueues
				,	per.MtDAgency
				,	UN.RiskBucket
				,	UN.EBAStatus
				,	eba.`PE/NPEDebtor`
				,	QCH.AssignmentChannel																								
				,	QCH.AssignmentTeam																									
				,	QCH.AssignmentAgent																									
				,	QCH.AssignmentAgentAM																									
				,	QCH.Queues																												
				,	QCH.CourseOfAction																									
				,	QCH.AssignmentSegmentDetailed
				,	per.Anchor
				,	SV.SavedFlag
				,	SV.SavedReason 
				,	SV.SavedDate
				,	CASE WHEN SV.SavedFlag = 'Saved' THEN AttrQCH_1.Queues END     AS SavedQueues
				,	CASE WHEN SV.SavedFlag = 'Saved' THEN AttrQCH_1.Agency END AS SavedAgency
				,	d.Treatmentbranch
				,	e.SubProduct
				,	AttrQCH.CourseOfAction AttributionCOA
				,	AttrQCH.AssignmentChannel AttributionChannel
				,	AttrQCH.AssignmentTeam AttributionTeam
				,	AttrQCH.AssignmentAgent AttributionAgent
				,	AttrQCH.AssignmentAgentAM AttributionAgentAM
				,	CAST(NULL AS VARCHAR(100)) AttributionAgent_2ndLevel
				,	CAST(NULL AS VARCHAR(100)) AttributionAgentAM_2ndLevel
				,	AttrQCH.AssignmentSegmentDetailed AttributionSegmentDetailed
				,	AttrQCH.Event
				,	AttrQCH.PaymentAmount
				,	TRIM(AttrQCH.AssignmentChannel) AttributionChannelOriginal
				,	TRIM(AttrQCH.AssignmentTeam) AttributionTeamOriginal
				,	TRIM(AttrQCH.AssignmentAgent) AttributionAgentOriginal
				,	TRIM(AttrQCH.AssignmentAgentAM) AttributionAgentAMOriginal
		FROM `{env_prefix}-cdh-delivery`.repoint.int_AssignedPerimeter  per 
		LEFT JOIN UnionFile      UN		ON per.ExposureKey = UN.ExposureNumber
		LEFT JOIN AFMTotalEBA	  eba		ON eba.Entity = per.Entity
		LEFT JOIN QueuesChannels QCH	ON per.ExposureKey = QCH.ExposureKey
		-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_DailyPortfolio D		ON D.ACCOUNT_NUMBER = per.ExposureKey
    	LEFT JOIN Treatmentbranch D	 ON D.ACCOUNT_NUMBER = per.ExposureKey
    	LEFT JOIN SubProduct E	 ON E.ACCOUNT_NUMBER = per.ExposureKey
		INNER JOIN AttrQueuesChannels_Rest AttrQCH	ON per.ExposureKey = AttrQCH.ExposureKey
		LEFT JOIN SavedInfo       SV	ON per.ExposureKey = SV.Account
		LEFT JOIN AttrQueuesChannels AttrQCH_1 ON per.ExposureKey = AttrQCH_1.ExposureKey 
''')
AssignedPerimeter_v1_0.createOrReplaceTempView("AssignedPerimeter_v1_0")   

In [0]:
AssignedPerimeter_v1 = spark.sql(
f'''	
SELECT DISTINCT     PortfolioName
				,	ExposureKey
				,	AgreementKey
				,	CDI
				,	PartyTaxRegistrationNumber
				,	Entity
				,	AssetSegment
				,	ProductKey
				,	SoMBalanceAmountEUR
				,	SoMSourceBucket
				,	SoMObjectiveDate
				,	SoMObjectiveStatus
				,   SoMDenouncedStatus
				,	MtDQueues
				,	MtDAgency
				,	RiskBucket
				,	EBAStatus
				,	`PE/NPEDebtor`
				,	AssignmentChannel																								
				,	AssignmentTeam																									
				,	AssignmentAgent		
				,	AssignmentAgentAM																							
				,	Queues																												
				,	CourseOfAction																									
				,	AssignmentSegmentDetailed
				,	Anchor
				,	SavedFlag
				,	SavedReason 
				,	SavedDate
				,	SavedQueues
				,	SavedAgency
				,	Treatmentbranch
				,	SubProduct
				,	AttributionCOA
				,	AttributionChannel
				,	CASE WHEN AttributionTeam IS NULL AND AttributionAgent IS NULL AND AttributionChannel  = 'HUB RMs' AND Event NOT IN ('Active Application','Saved due to Restructure','Saved due to Extension') AND AttributionAgent IS NULL THEN 'HUB RMs Rest'
						 WHEN AttributionAgent IS NULL AND AttributionChannel  = 'HUB RMs' AND Event NOT IN ('Active Application','Saved due to Restructure','Saved due to Extension') AND AttributionAgent IS NULL THEN CONCAT(AttributionTeam,'_Rest')
						 ELSE AttributionTeam END AttributionTeam
				,	AttributionAgent
				,	AttributionAgentAM
				,	AttributionAgent_2ndLevel
				,	AttributionAgentAM_2ndLevel
				,	AttributionSegmentDetailed
				,	Event
				,	PaymentAmount
				,	AttributionChannelOriginal
				,	AttributionTeamOriginal
				,	AttributionAgentOriginal
				,	AttributionAgentAMOriginal
		FROM AssignedPerimeter_v1_0	
''')
AssignedPerimeter_v1.createOrReplaceTempView("AssignedPerimeter_v1")    

##### UnionFileRefDates

In [0]:
UnionFileRefDates = spark.sql(
f'''			
SELECT   a.ExposureKey
        ,LAST_DAY(DATE_ADD(MONTH, -1, uf.FirstDateAtValidation)) RefDate
	    ,a.EBAStatus  AS SoMEBAStatus
		,l.`3rd level Assignment`
		,l.`2nd level Assignment`
		,l.`PE/NPE RM`
		,a.AttributionAgent
		,a.Event
FROM AssignedPerimeter_v1 a
INNER JOIN Report_Unified                               uf ON a.ExposureKey = uf.ExposureKey
LEFT JOIN assignmentlevelsmapping l   ON lower(l.`Full name 2`) = lower(a.AttributionAgent)
WHERE Event IN ('Restructure','Active Application','Saved due to Restructure')
		  AND AttributionChannel = 'HUB RMs'
		  AND uf.FirstDateAtValidation > '2023-06-30' AND uf.FirstDateAtValidation <= '2023-08-31'
		  AND a.EBAStatus IN ('NPL','FNPL')
''')
UnionFileRefDates.createOrReplaceTempView("UnionFileRefDates")       

##### EBAStatusAtApplicationDate

In [0]:
EBAStatusAtApplicationDate = spark.sql(
f'''		
SELECT   rf.*
		     , u.EBAStatus
			   , u.FPLUnderProbationFlag
FROM `{env_prefix}-cdh`.externalfiles.extunionfile  u
INNER JOIN UnionFileRefDates rf ON u.ExposureNumber = rf.ExposureKey AND u.SnapshotDate = rf.RefDate
INNER JOIN (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp, SnapshotDate FROM `{env_prefix}-cdh`.externalfiles.extunionfile GROUP BY SnapshotDate) IT ON IT.IngestionTimeStamp = U.IngestionTimeStamp AND IT.SnapshotDate = u.SnapshotDate
''')
EBAStatusAtApplicationDate.createOrReplaceTempView("EBAStatusAtApplicationDate")              

##### AssignmentTeam_NPE

In [0]:
AssignmentTeam_NPE = spark.sql(
f'''		
  SELECT DISTINCT `2nd level Assignment`
		FROM assignmentlevelsmapping
		WHERE `1st level Assignment`  = 'HUB RMs'
		  AND `PE/NPE RM` LIKE '%NPE%'
''')
AssignmentTeam_NPE.createOrReplaceTempView("AssignmentTeam_NPE")        

##### Activities_547_500_578_536

In [0]:
Activities_547_500_578_536 = spark.sql(
f'''		
SELECT   *
	   , RANK() OVER(PARTITION BY ExposureKey ORDER BY InteractionDate DESC, InteractionTypeKey ASC, CASE WHEN PartyRelationshipType = 'Primary Owner' THEN 1 ELSE 0 END DESC , CreatedDate DESC) AS RANK 
FROM (
		SELECT DISTINCT 
					A.ExposureKey 
				,	A.InteractionTypeKey 
				,	A.InteractionType 
				,	A.InteractionDate
				,	B.EmployeeFullName 	
				,	B.EmployeeUserName 
				,	nc.PartyRelationshipType
				, 	A.CreatedDate
		FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) A
		INNER JOIN `{env_prefix}-cdh`.primary.exposure nc on A.ExposureKey = nc.ExposureKey and nc.BusinessDate={run_date!r}
        LEFT JOIN `{env_prefix}-cdh`.primary.firmstructure B ON A.CreatedBy = B.EmployeePartyKey AND B.EmployeeCategory = "Employee" AND B.SourceSystem = 'qca' AND B.BusinessDate={run_date!r}
		WHERE A.InteractionTypeKey IN ('547','500','578','536')
		  AND A.InteractionDate <= {run_date!r} 	  
          AND A.PortfolioName = 'Core Retail'   
) C
''')
Activities_547_500_578_536.createOrReplaceTempView("Activities_547_500_578_536")        

In [0]:
Activities_547_500_578_536 = spark.sql(
f'''			
SELECT * 
FROM Activities_547_500_578_536
WHERE Rank = 1
''')
Activities_547_500_578_536.createOrReplaceTempView("Activities_547_500_578_536")    

In [0]:
Activities_524_533 = spark.sql(
f'''		
SELECT *
	 , RANK() OVER(PARTITION BY ExposureKey ORDER BY InteractionDate DESC, InteractionTypeKey ASC, CASE WHEN PartyRelationshipType = 'Primary Owner' THEN 1 ELSE 0 END DESC, CreatedDate DESC) AS RANK 
FROM (
		SELECT DISTINCT 
					A.ExposureKey 
				,	A.InteractionTypeKey
				,	A.InteractionType 
				,	A.InteractionDate 
				,	B.EmployeeFullName 
				,	B.EmployeeUserName 
				,	nc.PartyRelationshipType
				, 	A.CreatedDate
		FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) A
		INNER JOIN `{env_prefix}-cdh`.primary.exposure nc on A.ExposureKey = nc.ExposureKey and nc.BusinessDate={run_date!r}
        LEFT JOIN `{env_prefix}-cdh`.primary.firmstructure B ON A.CreatedBy = B.EmployeePartyKey AND B.EmployeeCategory = "Employee" AND B.SourceSystem = 'qca' AND B.BusinessDate={run_date!r}
		WHERE A.InteractionTypeKey IN ('524','533')
		  AND A.InteractionDate <= {run_date!r} 	  
          AND A.PortfolioName = 'Core Retail'
     ) C
''')
Activities_524_533.createOrReplaceTempView("Activities_524_533")        

In [0]:
Activities_524_533 = spark.sql(
f'''		
SELECT * 
FROM Activities_524_533
WHERE Rank = 1
''')
Activities_524_533.createOrReplaceTempView("Activities_524_533")    

##### AssignedPerimeter_v1 UPD

In [0]:
AssignedPerimeter_v1_upd1 = spark.sql(
f'''	
SELECT DISTINCT PortfolioName
				,	a.ExposureKey
				,	AgreementKey
				,	CDI
				,	PartyTaxRegistrationNumber
				,	Entity
				,	AssetSegment
				,	ProductKey
				,	SoMBalanceAmountEUR
				,	SoMSourceBucket
				,	SoMObjectiveDate
				,	SoMObjectiveStatus
				,   SoMDenouncedStatus
				,	MtDQueues
				,	MtDAgency
				,	RiskBucket
				,	a.EBAStatus
				,	`PE/NPEDebtor`
				,	AssignmentChannel																								
				,	AssignmentTeam																									
				,	AssignmentAgent	
				,	AssignmentAgentAM																								
				,	Queues																												
				,	CourseOfAction																									
				,	AssignmentSegmentDetailed
				,	Anchor
				,	SavedFlag
				,	SavedReason 
				,	SavedDate
				,	SavedQueues
				,	SavedAgency
				,	Treatmentbranch
				,	SubProduct
				,	AttributionCOA
				,	AttributionChannel
                , CASE WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND l.`PE/NPE RM` LIKE '%NPE%' THEN L.`2nd level Assignment`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND l.`PE/NPE RM` NOT LIKE '%NPE%' AND e.ExposureKey IS NOT NULL AND e.SoMEBAStatus IN ('PL','CFPL','FPL') THEN L.`2nd level Assignment`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND l.`PE/NPE RM` NOT LIKE '%NPE%' AND a.`PE/NPEDebtor` = 'PE' THEN L.`2nd level Assignment`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND np.`2nd level Assignment` IS NOT NULL THEN CONCAT(L.`2nd level Assignment` , '_Rest')
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND np.`2nd level Assignment` IS NULL THEN  CONCAT(a.Treatmentbranch , '_Rest')
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('PL','CFPL','FPL') AND a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' THEN L.`2nd level Assignment`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('PL','CFPL','FPL') AND a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' THEN L.`2nd level Assignment`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('PL','CFPL','FPL') AND a.Event IN('Saved due to Extension','Extension') AND a.AttributionChannel = 'HUB RMs' THEN L.`2nd level Assignment`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') THEN AttributionTeamOriginal ELSE AttributionTeam END AS AttributionTeam
                ,   CASE WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND l.`PE/NPE RM` like '%NPE%' THEN L.`Full name` 
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND l.`PE/NPE RM` NOT LIKE '%NPE%' AND e.ExposureKey IS NOT NULL AND e.SoMEBAStatus IN ('PL','CFPL','FPL') THEN L.`Full name`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel= 'HUB RMs' AND l.`PE/NPE RM` NOT LIKE '%NPE%' AND a.`PE/NPEDebtor` = 'PE' THEN L.`Full name` 
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND np.`2nd level Assignment` IS NOT NULL THEN NULL
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND np.`2nd level Assignment` IS NULL THEN  NULL
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('PL','CFPL','FPL') AND a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' THEN L.`Full name` 
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('PL','CFPL','FPL') AND a.Event IN ('Saved due to Extension','Extension') AND a.AttributionChannel = 'HUB RMs' THEN L.`Full name` 
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') THEN AttributionAgentOriginal ELSE a.AttributionAgent END AS AttributionAgent
                ,   CASE WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND l.`PE/NPE RM` like '%NPE%' THEN L.`3rd level Assignment`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND l.`PE/NPE RM` NOT LIKE '%NPE%' AND e.ExposureKey IS NOT NULL AND e.SoMEBAStatus IN ('PL','CFPL','FPL') THEN L.`3rd level Assignment` 
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND l.`PE/NPE RM` NOT LIKE '%NPE%' AND a.`PE/NPEDebtor` = 'PE' THEN L.`3rd level Assignment`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND np.`2nd level Assignment` IS NOT NULL THEN NULL
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('NPL','FNPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' AND np.`2nd level Assignment` IS NULL THEN  NULL
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('PL','CFPL','FPL') AND a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel = 'HUB RMs' THEN L.`3rd level Assignment`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') AND a.EBAStatus IN ('PL','CFPL','FPL') AND a.Event IN('Saved due to Extension','Extension') AND a.AttributionChannel = 'HUB RMs' THEN L.`3rd level Assignment`
					   WHEN a.AttributionChannel = 'HUB RMs' AND a.Event IN('Restructure','Active Application','Saved due to Restructure','Saved due to Extension', 'Extension') THEN AttributionAgentAMOriginal
				  ELSE AttributionAgentAM END AS AttributionAgentAM 
				,	AttributionAgent_2ndLevel
				,	AttributionAgentAM_2ndLevel
				,	AttributionSegmentDetailed
				,	a.Event
				,	PaymentAmount
				,	AttributionChannelOriginal
				,	AttributionTeamOriginal
				,	AttributionAgentOriginal
				,	AttributionAgentAMOriginal
		FROM AssignedPerimeter_v1	a
  		LEFT JOIN EBAStatusAtApplicationDate e ON a.ExposureKey = e.ExposureKey
        LEFT JOIN assignmentlevelsmapping l   ON lower(l.`3rd level Assignment`) = lower(a.AttributionAgentAM)
		LEFT JOIN AssignmentTeam_NPE np ON lower(np.`2nd level Assignment`) = lower(l.`2nd level Assignment`)
''')
AssignedPerimeter_v1_upd1.createOrReplaceTempView("AssignedPerimeter_v1_upd1") 

In [0]:
AssignedPerimeter_v1_upd2 = spark.sql(
f'''	
SELECT DISTINCT     PortfolioName
				,	a.ExposureKey
				,	AgreementKey
				,	CDI
				,	PartyTaxRegistrationNumber
				,	Entity
				,	AssetSegment
				,	ProductKey
				,	SoMBalanceAmountEUR
				,	SoMSourceBucket
				,	SoMObjectiveDate
				,	SoMObjectiveStatus
				,   SoMDenouncedStatus
				,	MtDQueues
				,	MtDAgency
				,	RiskBucket
				,	a.EBAStatus
				,	`PE/NPEDebtor`
				,	AssignmentChannel																								
				,	AssignmentTeam																									
				,	AssignmentAgent		
				,	AssignmentAgentAM																							
				,	Queues																												
				,	CourseOfAction																									
				,	AssignmentSegmentDetailed
				,	Anchor
				,	SavedFlag
				,	SavedReason 
				,	SavedDate
				,	SavedQueues
				,	SavedAgency
				,	Treatmentbranch
				,	SubProduct
				,	AttributionCOA
				,	AttributionChannel
                ,  CASE WHEN a.Event IN ('Saved due to Extension','Extension') AND a.AttributionChannel IN('IRT','Call Center') THEN 'IRT w/o UWF Application'
				  ELSE AttributionTeam END AS AttributionTeam
                , CASE WHEN a.Event IN('Saved due to Extension','Extension') AND a.AttributionChannel IN('IRT','Call Center') THEN IFNULL(L.`Full name`, a.AttributionAgentOriginal) ELSE a.AttributionAgent END AS AttributionAgent
                ,   CASE WHEN a.Event IN('Saved due to Extension','Extension') AND a.AttributionChannel IN('IRT','Call Center') THEN IFNULL(L.`3rd level Assignment`, a.AttributionAgentAMOriginal) ELSE AttributionAgentAM END AS AttributionAgentAM
				,	AttributionAgent_2ndLevel
				,	AttributionAgentAM_2ndLevel
				,	AttributionSegmentDetailed
				,	a.Event
				,	PaymentAmount
				,	AttributionChannelOriginal
				,	AttributionTeamOriginal
				,	AttributionAgentOriginal
				,	AttributionAgentAMOriginal
		FROM AssignedPerimeter_v1_upd1	a
        LEFT JOIN assignmentlevelsmapping l   ON lower(l.`3rd level Assignment`) = lower(a.AttributionAgentAM)
''')
AssignedPerimeter_v1_upd2.createOrReplaceTempView("AssignedPerimeter_v1_upd2") 

In [0]:
AssignedPerimeter_v1_upd3 = spark.sql(
f'''	
SELECT DISTINCT     PortfolioName
				,	a.ExposureKey
				,	AgreementKey
				,	CDI
				,	PartyTaxRegistrationNumber
				,	Entity
				,	AssetSegment
				,	ProductKey
				,	SoMBalanceAmountEUR
				,	SoMSourceBucket
				,	SoMObjectiveDate
				,	SoMObjectiveStatus
				,   SoMDenouncedStatus
				,	MtDQueues
				,	MtDAgency
				,	RiskBucket
				,	a.EBAStatus
				,	`PE/NPEDebtor`
				,	AssignmentChannel																								
				,	AssignmentTeam																									
				,	AssignmentAgent	
				,	AssignmentAgentAM																								
				,	Queues																												
				,	CourseOfAction																									
				,	AssignmentSegmentDetailed
				,	Anchor
				,	SavedFlag
				,	SavedReason 
				,	SavedDate
				,	SavedQueues
				,	SavedAgency
				,	Treatmentbranch
				,	SubProduct
				,	AttributionCOA
				,	AttributionChannel
                ,   CASE WHEN a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel IN ('IRT','Call Center') THEN 'IRT BAU' ELSE AttributionTeam END AS AttributionTeam
                ,   CASE WHEN a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel IN ('IRT','Call Center') THEN IFNULL(l1.`Full name`, act1.EmployeeFullName) ELSE a.AttributionAgent END AS AttributionAgent
                ,   CASE WHEN a.Event IN ('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel IN ('IRT','Call Center')	THEN IFNULL(l1.`3rd level Assignment`, act1.EmployeeUserName) ELSE AttributionAgentAM END AS AttributionAgentAM
				,	CASE WHEN a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel IN ('IRT','Call Center')	THEN IFNULL(l2.`Full name`, act2.EmployeeFullName) ELSE AttributionAgent_2ndLevel END AS AttributionAgent_2ndLevel
				,	CASE WHEN a.Event IN('Restructure','Active Application','Saved due to Restructure') AND a.AttributionChannel IN ('IRT','Call Center')	THEN IFNULL(l2.`3rd level Assignment`, act2.EmployeeUserName) ELSE AttributionAgentAM_2ndLevel END AS AttributionAgentAM_2ndLevel
				,	AttributionSegmentDetailed
				,	a.Event
				,	PaymentAmount
				,	AttributionChannelOriginal
				,	AttributionTeamOriginal
				,	AttributionAgentOriginal
				,	AttributionAgentAMOriginal
		FROM AssignedPerimeter_v1_upd2	a
		LEFT JOIN Activities_547_500_578_536 act1 ON act1.ExposureKey = a.ExposureKey
		LEFT JOIN Activities_524_533 act2 ON act2.ExposureKey = a.ExposureKey
    LEFT JOIN assignmentlevelsmapping l1   ON lower(l1.`3rd level Assignment`) = lower(act1.EmployeeUserName)
    LEFT JOIN assignmentlevelsmapping l2   ON lower(l2.`3rd level Assignment`) = lower(act2.EmployeeUserName)
    LEFT JOIN assignmentlevelsmapping l3   ON lower(l3.`3rd level Assignment`) = lower(a.AttributionAgentAM) 
''')
AssignedPerimeter_v1_upd3.createOrReplaceTempView("AssignedPerimeter_v1_upd3")

##### Mods_Historic

In [0]:
HistoricalModificationBefore202309 = spark.sql(
f'''
SELECT DISTINCT h.OldExposureKey, h.NewExposureKey, h.RestructuringDate
FROM `{env_prefix}-cdh`.reference.corestatichistoricalmodificationbeforemigration h
WHERE h.OldExposureKey <> h.NewExposureKey AND h.OldExposureKey <> '' AND h.NewExposureKey <> ''
''')
HistoricalModificationBefore202309.createOrReplaceTempView("HistoricalModificationBefore202309")  


ModificationFullMapping = spark.sql(
f'''
SELECT 
						 NewExposureKey, 
						 OldExposureKey,
                         RestructuringDate
FROM HistoricalModificationBefore202309 h
UNION ALL
SELECT 
						 M.NewExposureKey, 
						 A.OldExposureKey,
                         A.RestructuringDate
FROM HistoricalModificationBefore202309 A
JOIN HistoricalModificationBefore202309 M
ON M.OldExposureKey = A.NewExposureKey
AND M.NewExposureKey <> A.OldExposureKey
''')
ModificationFullMapping.createOrReplaceTempView("ModificationFullMapping") 


extlpfxb364_ever = spark.sql(
f'''
SELECT 
    OldExposureKey,
    NewExposureKey,
    CASE WHEN SUBSTRING(RestructuringDate, 5, 1) = '-' THEN try_cast(RestructuringDate AS date)
        WHEN SUBSTRING(RestructuringDate, 3, 1) = '/' THEN TO_DATE(RestructuringDate, 'dd/MM/yyyy')
        END AS RestructuringDate,
    RestructuringType,
    AgreementKey,
    BusinessDate
FROM  `{env_prefix}-cdh`.externalfiles.extlpfxb364
''')
extlpfxb364_ever.createOrReplaceTempView("extlpfxb364_ever")


Mods_Historic = spark.sql('''
select distinct
    OldExposureKey,
    RestructuringDate 
from extlpfxb364_ever
union all
select distinct
    OldExposureKey,
    RestructuringDate 
from ModificationFullMapping
''')

Mods_Historic.createOrReplaceTempView("Mods_Historic")


##### AssignedPerimeter_v1 new UPD

In [0]:
AssignedPerimeter_v1_upd4 = spark.sql(
f'''	
SELECT DISTINCT 
          PortfolioName
				,	ExposureKey
				,	AgreementKey
				,	CDI
				,	PartyTaxRegistrationNumber
				,	Entity
				,	AssetSegment
				,	ProductKey
				,	SoMBalanceAmountEUR
				,	SoMSourceBucket
				,	SoMObjectiveDate
				,	SoMObjectiveStatus
				, 	SoMDenouncedStatus
				,	MtDQueues
				,	MtDAgency
				,	RiskBucket
				,	EBAStatus
				,	`PE/NPEDebtor`
				,	AssignmentChannel																								
				,	AssignmentTeam																									
				,	AssignmentAgent	
				,	AssignmentAgentAM																								
				,	Queues																												
				,	CourseOfAction																									
				,	AssignmentSegmentDetailed
				,	Anchor
				,	SavedFlag
				,	SavedReason 
				,	SavedDate
				,	SavedQueues
				,	SavedAgency
				,	Treatmentbranch
				,	SubProduct
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionCOA END AS AttributionCOA
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionChannel END AS AttributionChannel
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionTeam END AS AttributionTeam
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionAgent END AS AttributionAgent
        , CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionAgentAM END AS AttributionAgentAM
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionAgent_2ndLevel END AS AttributionAgent_2ndLevel
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionAgentAM_2ndLevel END AS AttributionAgentAM_2ndLevel
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionSegmentDetailed END AS AttributionSegmentDetailed
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE Event END AS Event
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE PaymentAmount END AS PaymentAmount
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionChannelOriginal END AS AttributionChannelOriginal
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionTeamOriginal END AS AttributionTeamOriginal
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionAgentOriginal END AS AttributionAgentOriginal
				,	CASE WHEN m.RestructuringDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r})) AND m.RestructuringDate > LAST_DAY(DATE_ADD(MONTH, -6, {run_date!r})) THEN NULL ELSE AttributionAgentAMOriginal END AS AttributionAgentAMOriginal
		FROM AssignedPerimeter_v1_upd3	a
		LEFT JOIN Mods_Historic m on a.ExposureKey = m.OldExposureKey 
''')
AssignedPerimeter_v1_upd4.createOrReplaceTempView("AssignedPerimeter_v1_upd4")

In [0]:
AssignedPerimeter_v1_upd5 = spark.sql(
f'''	
SELECT DISTINCT     PortfolioName
				,	ExposureKey
				,	AgreementKey
				,	CDI
				,	PartyTaxRegistrationNumber
				,	Entity
				,	AssetSegment
				,	ProductKey
				,	SoMBalanceAmountEUR
				,	SoMSourceBucket
				,	SoMObjectiveDate
				,	SoMObjectiveStatus
				,   SoMDenouncedStatus
				,	MtDQueues
				,	MtDAgency
				,	RiskBucket
				,	EBAStatus
				,	`PE/NPEDebtor`
				,	AssignmentChannel																								
				,	CASE WHEN AssignmentChannel = 'IRT' AND AssignmentTeam like 'PRE_%' THEN 'Unassigned' ELSE AssignmentTeam END AS AssignmentTeam
				,	AssignmentAgent	
				,	AssignmentAgentAM																								
				,	Queues																												
				,	CourseOfAction																									
				,	AssignmentSegmentDetailed
				,	Anchor
				,	SavedFlag
				,	SavedReason 
				,	SavedDate
				,	SavedQueues
				,	SavedAgency
				,	Treatmentbranch
				,	SubProduct
				,	CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionCOA END AS AttributionCOA
				,	CASE WHEN AttributionChannel = 'IRT' AND AttributionTeam like 'PRE_%' THEN 'Unassigned'
                         WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionChannel END AS AttributionChannel
                ,   CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionTeam END AS AttributionTeam
                ,   CASE WHEN AttributionAgent LIKE '%INTERNAL%' THEN NULL
                         WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionAgent END AS AttributionAgent
                ,   CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionAgentAM END AS AttributionAgentAM
				,	CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionAgent_2ndLevel END AS AttributionAgent_2ndLevel
				,	CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionAgentAM_2ndLevel END AS AttributionAgentAM_2ndLevel
				,	CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionSegmentDetailed END AS AttributionSegmentDetailed
				,	CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE Event END AS Event
				,	CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE PaymentAmount END AS PaymentAmount
				,	CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionChannelOriginal END AS AttributionChannelOriginal
				,	CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionTeamOriginal END AS AttributionTeamOriginal
				,	CASE WHEN AttributionAgentOriginal ILIKE '%INTERNAL%' THEN NULL
                         WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionAgentOriginal END AS AttributionAgentOriginal
				,	CASE WHEN EBAStatus = 'CFPL' THEN NULL ELSE AttributionAgentAMOriginal END AS AttributionAgentAMOriginal
		FROM AssignedPerimeter_v1_upd4
''')
AssignedPerimeter_v1_upd5.createOrReplaceTempView("AssignedPerimeter_v1_upd5")

In [0]:
AssignedPerimeter_v1_upd6 = spark.sql(
f'''	
SELECT DISTINCT     {run_date!r} BusinessDate
        		, 	PortfolioName
				,	ExposureKey
				,	AgreementKey
				,	CDI
				,	PartyTaxRegistrationNumber
				,	Entity
				,	AssetSegment
				,	ProductKey
				,	SoMBalanceAmountEUR
				,	SoMSourceBucket
				,	SoMObjectiveDate
				,	SoMObjectiveStatus
				, 	SoMDenouncedStatus
				,	MtDQueues
				,	MtDAgency
				,	RiskBucket
				,	EBAStatus
				,	`PE/NPEDebtor`
				,	AssignmentChannel																								
				,	AssignmentTeam																									
				,	AssignmentAgent	
				,	AssignmentAgentAM
				,	Queues																												
				,	CASE WHEN CourseOfAction = 'unhandled' THEN CourseOfAction
                         WHEN CourseOfAction = 'N.3869' THEN CourseOfAction
                         WHEN AssignmentChannel = 'Legal Agencies 3869' THEN 'N.3869'
                         WHEN AssignmentChannel = 'Outsourcing Collection' THEN 'Collection'
						 WHEN AssignmentChannel IN ('HUB RMs','IRT','Legal Agencies') AND EBAStatus IN ('PL','CFPL','FPL') THEN 'Double Treatment'
						 WHEN AssignmentChannel = 'HUB RMs / Call Center' THEN 'Double Treatment'
						 WHEN AssignmentChannel IN ('HUB RMs','Call Center','Legal Agencies') AND 
									     (IFNULL(MtDQueues, SavedQueues) LIKE '%DT%' OR  IFNULL(MtDQueues, SavedQueues) LIKE '%Dou%') THEN 'Double Treatment'
						 WHEN AssignmentChannel IN ('HUB RMs','IRT','Legal Agencies')	 THEN 'Restructuring'
						 ELSE CourseOfAction END CourseOfAction 
				,	AssignmentSegmentDetailed
				,	Anchor
				,	SavedFlag
				,	SavedReason 
				,	SavedDate
				,	SavedQueues
				,	SavedAgency
				,	Treatmentbranch
				,	SubProduct
				,	CASE WHEN AttributionCOA = 'unhandled' THEN AttributionCOA
                         WHEN AttributionCOA = 'N.3869' THEN AttributionCOA
                         WHEN AttributionChannel = 'Legal Agencies 3869' THEN 'N.3869'
                         WHEN AttributionChannel = 'Outsourcing Collection' THEN 'Collection'
						 WHEN AttributionChannel IN ('HUB RMs','IRT','Legal Agencies') AND EBAStatus IN ('PL','CFPL','FPL') THEN 'Double Treatment'
						 WHEN AttributionChannel = 'HUB RMs / Call Center' THEN 'Double Treatment'
						 WHEN AttributionChannel IN ('HUB RMs','Call Center','Legal Agencies') AND 
									     (IFNULL(MtDQueues, SavedQueues) LIKE '%DT%' OR  IFNULL(MtDQueues, SavedQueues) LIKE '%Dou%') THEN 'Double Treatment'
						 WHEN AttributionChannel IN ('HUB RMs','IRT','Legal Agencies') THEN 'Restructuring'
						 ELSE AttributionCOA
						 END AttributionCOA
				,	AttributionChannel
				,	AttributionTeam
				,	AttributionAgent
        		, 	AttributionAgentAM
				,	AttributionAgent_2ndLevel
				,	AttributionAgentAM_2ndLevel
				,	AttributionSegmentDetailed
				,	Event
				,	PaymentAmount
				,	AttributionChannelOriginal
				,	AttributionTeamOriginal
				,	AttributionAgentOriginal
				,	AttributionAgentAMOriginal
		FROM AssignedPerimeter_v1_upd5
''')
AssignedPerimeter_v1_upd6.createOrReplaceTempView("AssignedPerimeter_v1_upd6")

In [0]:
final_df = spark.sql(
f'''	
SELECT DISTINCT     CAST(BusinessDate 				AS DATE) AS BusinessDate
        		, 	CAST(PortfolioName 				AS STRING) AS PortfolioName
				,	CAST(ExposureKey 				AS STRING) AS ExposureKey
				,	CAST(AgreementKey 				AS STRING) AS AgreementKey
				,	CAST(CDI 						AS STRING) AS CDI
				,	CAST(PartyTaxRegistrationNumber AS STRING) AS PartyTaxRegistrationNumber
				,	CAST(Entity 					AS STRING) AS Entity
				,	CAST(AssetSegment 				AS STRING) AS AssetSegment
				,	CAST(ProductKey 				AS STRING) AS ProductKey
				,	CAST(SoMBalanceAmountEUR 		AS DECIMAL(19,2)) AS SoMBalanceAmountEUR
				,	CAST(SoMSourceBucket 			AS INT) AS SoMSourceBucket
				,	CAST(SoMObjectiveDate 			AS DATE) AS SoMObjectiveDate
				,	CAST(SoMObjectiveStatus 		AS STRING) AS SoMObjectiveStatus
				, 	CAST(SoMDenouncedStatus 		AS STRING) AS SoMDenouncedStatus
				,	CAST(MtDQueues 					AS STRING) AS MtDQueues
				,	CAST(MtDAgency 					AS STRING) AS MtDAgency
				,	CAST(RiskBucket 				AS INT) AS RiskBucket
				,	CAST(EBAStatus 					AS STRING) AS EBAStatus
				,	CAST(`PE/NPEDebtor` 			AS STRING) AS PENPEDebtor
				,	CAST(AssignmentChannel 			AS STRING) AS AssignmentChannel
				,	CAST(AssignmentTeam 			AS STRING) AS AssignmentTeam
				,	CAST(AssignmentAgent 			AS STRING) AS AssignmentAgent
				,	CAST(AssignmentAgentAM 			AS STRING) AS AssignmentAgentAM
				,	CAST(Queues 					AS STRING) AS Queues
				,	CAST(CourseOfAction 			AS STRING) AS CourseOfAction
				,	CAST(AssignmentSegmentDetailed  AS STRING) AS AssignmentSegmentDetailed
				,	CAST(SavedFlag 					AS STRING) AS SavedFlag
				,	CAST(SavedReason 				AS STRING) AS SavedReason
				,	CAST(SavedDate 					AS DATE) AS SavedDate
				,	CAST(SavedAgency 				AS STRING) AS SavedAgency
				,	CAST(Treatmentbranch 			AS STRING) AS Treatmentbranch
				,	CAST(SubProduct 				AS STRING) AS SubProduct
				,	CAST(AttributionCOA 			AS STRING) AS AttributionCOA
				,	CAST(AttributionChannel 		AS STRING) AS AttributionChannel
				,	CAST(AttributionTeam 			AS STRING) AS AttributionTeam
				,	CAST(AttributionAgent 			AS STRING) AS AttributionAgent
				,	CAST(AttributionAgentAM 		AS STRING) AS AttributionAgentAM
				,	CAST(AttributionAgent_2ndLevel  AS STRING) AS AttributionAgent_2ndLevel
				,	CAST(AttributionAgentAM_2ndLevel AS STRING) AS AttributionAgentAM_2ndLevel
				,	CAST(AttributionSegmentDetailed AS STRING) AS AttributionSegmentDetailed
				,	CAST(Event 						AS STRING) AS Event
				,	CAST(IFNULL(PaymentAmount, 0)   AS DECIMAL(19,2)) AS PaymentAmount
				,	CAST(AttributionChannelOriginal AS STRING) AS AttributionChannelOriginal
				,	CAST(AttributionTeamOriginal 	AS STRING) AS AttributionTeamOriginal
				,	CAST(AttributionAgentOriginal 	AS STRING) AS AttributionAgentOriginal
				,	CAST(AttributionAgentAMOriginal AS STRING) AS AttributionAgentAMOriginal
				,	CAST(getdate()					AS TIMESTAMP) AS InsertTimeStamp
		FROM AssignedPerimeter_v1_upd6
''')
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_AssignmentDataMapping: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[coalesce(col(c), lit("''")) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey", "Event", "PaymentAmount", "AttributionChannelOriginal", "AttributionTeamOriginal", "AttributionAgentOriginal", "AttributionSegmentDetailed", "AttributionCOA"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
AssignmentDataMapping = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
AssignmentDataMapping.createOrReplaceTempView("AssignmentDataMapping")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/AssignmentDataMapping/"
file_name = "AssignmentDataMapping_" +run_date+ ".csv"
AssignmentDataMapping.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)